# Bored Pile Settlement Prediction — LSSVM, LSSVM-PSO, LSSVM-BO, LSSVM-GWO

**Workflow**
1. Setup & configuration
2. Dataset analysis — statistics, correlation heatmap with KDE, violin plots
3. Train/test split + K-fold cross-validation scheme
4. LSSVM model (RBF kernel)
5. Hyperparameter optimisation — PSO, Bayesian optimisation (BO), GWO (convergence & error curves)
6. Final training, saving hyperparameters & trained models
7. K-fold CV of final models + learning (error) curves
8. Model evaluation — 8 metrics (R², RMSE, MAE, MAPE, VAF, WI, a20-index, PI) & rank analysis
9. Performance analysis — regression plots, Taylor diagram, comparison charts
10. Error & residual analysis
11. Sensitivity analysis of the best model (CAM, permutation, OAT, SHAP)

All outputs go to `Results_LSSVM/` in separate sub-folders. Figures are saved at 600 dpi (PNG) + vector PDF.

## 1. Setup & configuration

In [ ]:
import os, json, time, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects
import matplotlib.font_manager as fm
from matplotlib.ticker import AutoMinorLocator, LogLocator, FuncFormatter, NullFormatter, MaxNLocator
import seaborn as sns
import joblib
from scipy import stats
from scipy.linalg import solve
from scipy.stats import qmc, norm as normal_dist
from scipy.optimize import minimize
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, KFold,
                                     GroupKFold, cross_validate, learning_curve)
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel, WhiteKernel
warnings.filterwarnings("ignore")

# ------------------------- USER CONFIGURATION -------------------------
# ---- data ----
DATA_FILE   = r"Raw data.xlsx"   # path to your data file
SHEET       = "Sheet1"
TARGET      = "s"                 # output column (settlement)
TEST_SIZE   = 0.20
SEED        = 42
K_FOLDS     = 5

# 'random' : rows shuffled randomly (common in the literature)
# 'group'  : whole piles (load-settlement curves) kept together in train OR test
#            -> stricter, avoids leakage between points of the same pile
SPLIT_MODE  = "random"

# Metaheuristic settings (same budget for all optimisers = fair comparison)
POP_SIZE    = 25      # PSO / GWO population size; BO uses POP_SIZE initial (Latin hypercube) points
MAX_ITER    = 200     # iterations (BO: one new evaluation per iteration)
# Search space in log10 scale: gamma (regularisation) and sigma (RBF width)
LOG_BOUNDS  = {"gamma": (-1.0, 6.0),     # gamma in [0.1, 10^6]
               "sigma": (-2.0, 1.5)}     # sigma in [0.01, ~31.6]
DEFAULT_LSSVM = {"gamma": 1.0, "sigma": 1.0}   # un-optimised base LSSVM

PLOT_LANDSCAPE = True     # CV-RMSE fitness landscape (grid search, ~1-3 min)
LANDSCAPE_GRID = 20
DPI = 600

# Axis labels (edit to add units, e.g. "D (m)", "Q (kN)", "s (mm)")
LABELS = {"D": "D", "L": "L", "R_c": "R$_c$", "N_c_avg": "N$_{c,avg}$",
          "N_s_avg": "N$_{s,avg}$", "N_tip": "N$_{tip}$", "I_tc": "I$_{tc}$",
          "sigma_v_tip": "$\\sigma'_{v,tip}$", "Q": "Q", "s": "s"}
# ----------------------------------------------------------------------

np.random.seed(SEED)

# Output folders
ROOT = "Results_LSSVM"
DIRS = {
    "data":   f"{ROOT}/01_Dataset",
    "eda":    f"{ROOT}/02_Figures/01_EDA",
    "opt":    f"{ROOT}/02_Figures/02_Optimization",
    "cv":     f"{ROOT}/02_Figures/03_CrossValidation",
    "perf":   f"{ROOT}/02_Figures/04_Performance",
    "reg":    f"{ROOT}/02_Figures/05_Regression",
    "err":    f"{ROOT}/02_Figures/06_Error_Residual",
    "sens":   f"{ROOT}/02_Figures/07_Sensitivity",
    "models": f"{ROOT}/03_Trained_Models",
    "hyper":  f"{ROOT}/04_Hyperparameters",
    "pred":   f"{ROOT}/05_Predictions",
    "metric": f"{ROOT}/06_Metrics",
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

# ---------------- Figure style: Times New Roman, publication quality ----------------
if "Times New Roman" not in {f.name for f in fm.fontManager.ttflist}:
    print("WARNING: Times New Roman not found — a fallback serif font will be used.")
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "custom", "mathtext.rm": "Times New Roman",
    "mathtext.it": "Times New Roman:italic", "mathtext.bf": "Times New Roman:bold",
    "font.size": 13, "axes.labelsize": 14, "axes.titlesize": 14, "legend.fontsize": 11,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "axes.linewidth": 1.2,
    "xtick.direction": "in", "ytick.direction": "in", "xtick.top": True, "ytick.right": True,
    "xtick.major.size": 5, "ytick.major.size": 5, "xtick.minor.size": 2.5, "ytick.minor.size": 2.5,
    "xtick.major.width": 1.1, "ytick.major.width": 1.1,
    "legend.frameon": True, "legend.edgecolor": "0.2", "legend.fancybox": False, "legend.framealpha": 0.95,
    "lines.linewidth": 2.0, "savefig.dpi": DPI, "figure.dpi": 110, "savefig.bbox": "tight",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})
MODEL_COLORS  = {"LSSVM": "#4C72B0", "LSSVM-PSO": "#DD8452", "LSSVM-BO": "#8172B3", "LSSVM-GWO": "#C44E52"}
MODEL_MARKERS = {"LSSVM": "o", "LSSVM-PSO": "s", "LSSVM-BO": "^", "LSSVM-GWO": "D"}
OPTIMISERS = ["PSO", "GWO"]          # add "BO" to include Bayesian optimisation
OPT_COLORS = {k: MODEL_COLORS[f"LSSVM-{k}"] for k in OPTIMISERS}

def save_fig(fig, folder, name):
    """Save a figure as 600-dpi PNG and vector PDF (unused, empty grid panels are hidden)."""
    empty = [ax for ax in fig.axes if not ax.has_data() and not ax.get_title(loc="left") and not ax.texts
             and ax.name == "rectilinear"]
    if empty:
        for ax in empty: ax.set_visible(False)
        for ax in fig.axes:
            if ax.get_visible() and ax.name == "rectilinear": ax.tick_params(labelbottom=True)
    fig.savefig(f"{DIRS[folder]}/{name}.png", dpi=DPI)
    fig.savefig(f"{DIRS[folder]}/{name}.pdf")
    plt.show()
    plt.close(fig)

def style_ax(ax, logy=False):
    """Light grid + minor ticks for a clean research look."""
    ax.grid(True, which="major", ls="--", lw=0.6, color="0.8", zorder=0)
    ax.set_axisbelow(True)
    if logy:   # plain numbers (0.8, 1, 2, 3 ...) instead of 2×10^0
        ax.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 3.0, 5.0)))
        ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(1, 10)))
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_formatter(NullFormatter())
    else:
        ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.xaxis.set_minor_locator(AutoMinorLocator())

print("Output folder:", os.path.abspath(ROOT))

## 2. Dataset analysis
### 2.1 Load data & descriptive statistics

In [ ]:
df = pd.read_excel(DATA_FILE, sheet_name=SHEET)
df = df.dropna().reset_index(drop=True)
FEATURES = [c for c in df.columns if c != TARGET]
print("Shape:", df.shape)
print("Inputs:", FEATURES, "| Output:", TARGET)

# Identify piles (rows sharing identical pile/soil properties = one load-settlement curve)
pile_cols = [c for c in FEATURES if c != "Q"]
df["Pile_ID"] = df.groupby(pile_cols, sort=False).ngroup() + 1
print("Number of piles (load-settlement curves):", df["Pile_ID"].nunique())

desc = df[FEATURES + [TARGET]].describe().T
desc["skewness"] = df[FEATURES + [TARGET]].skew()
desc["kurtosis"] = df[FEATURES + [TARGET]].kurt()
desc["CV (%)"]   = desc["std"] / desc["mean"] * 100
desc = desc.rename(columns={"25%": "Q1 (25%)", "50%": "Median", "75%": "Q3 (75%)"})
desc.to_excel(f"{DIRS['data']}/Dataset_Statistics.xlsx")
shutil.copy(DATA_FILE, f"{DIRS['data']}/Raw_data_copy.xlsx")
desc.round(3)

### 2.2 Correlation matrix (Pearson & Spearman heatmaps)

In [ ]:
cols = FEATURES + [TARGET]
lab = [LABELS.get(c, c) for c in cols]
corr_p = df[cols].corr(method="pearson")
corr_s = df[cols].corr(method="spearman")
with pd.ExcelWriter(f"{DIRS['data']}/Correlation_Matrix.xlsx") as w:
    corr_p.to_excel(w, sheet_name="Pearson")
    corr_s.to_excel(w, sheet_name="Spearman")

for name, cm in [("Pearson", corr_p), ("Spearman", corr_s)]:
    fig, ax = plt.subplots(figsize=(8.5, 7))
    mask = np.triu(np.ones_like(cm, dtype=bool), k=1)
    sns.heatmap(cm, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
                square=True, linewidths=0.6, linecolor="white", cbar_kws={"shrink": 0.8,
                "label": f"{name} correlation coefficient"}, annot_kws={"size": 9},
                xticklabels=lab, yticklabels=lab, ax=ax)
    ax.tick_params(top=False, right=False)
    plt.xticks(rotation=45, ha="right")
    save_fig(fig, "eda", f"Fig_Correlation_Heatmap_{name}")

### 2.3 Correlation heatmap matrix with KDE (pair-grid)
Lower triangle: scatter + bivariate KDE contours · Diagonal: histogram + KDE · Upper triangle: Pearson *r* coloured by strength.

In [ ]:
cmap = plt.get_cmap("RdBu_r")
norm = matplotlib.colors.Normalize(vmin=-1, vmax=1)

def corr_upper(x, y, **kw):
    r, p = stats.pearsonr(x, y)
    ax = plt.gca()
    ax.set_facecolor(cmap(norm(r)))
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    ax.annotate(f"{r:.2f}{stars}", xy=(0.5, 0.5), xycoords="axes fraction", ha="center",
                va="center", fontsize=11 + 6 * abs(r), fontweight="bold",
                color="white" if abs(r) > 0.6 else "black")
    ax.set_xticks([]); ax.set_yticks([])

def kde_lower(x, y, **kw):
    ax = plt.gca()
    ax.scatter(x, y, s=4, color="0.35", alpha=0.35, lw=0)
    try:
        sns.kdeplot(x=x, y=y, levels=6, fill=True, cmap="viridis", alpha=0.55, ax=ax, warn_singular=False)
    except Exception:
        pass

def diag_hist(x, **kw):
    ax = plt.gca()
    sns.histplot(x, kde=True, stat="density", color="#4C72B0", edgecolor="white",
                 line_kws={"lw": 1.5}, ax=ax)
    ax.lines[-1].set_color("#C44E52") if ax.lines else None

g = sns.PairGrid(df[cols], diag_sharey=False, height=1.55)
g.map_upper(corr_upper)
g.map_lower(kde_lower)
g.map_diag(diag_hist)
for i, c in enumerate(cols):
    g.axes[-1, i].set_xlabel(LABELS.get(c, c), fontsize=12)
    g.axes[i, 0].set_ylabel(LABELS.get(c, c), fontsize=12)
for ax in g.axes.flat:
    ax.tick_params(labelsize=7, top=False, right=False)
sm = matplotlib.cm.ScalarMappable(cmap=cmap, norm=norm)
cax = g.fig.add_axes([1.01, 0.3, 0.015, 0.4])
g.fig.colorbar(sm, cax=cax, label="Pearson r")
g.fig.text(0.5, -0.01, "* p<0.05, ** p<0.01, *** p<0.001", ha="center", fontsize=10)
save_fig(g.fig, "eda", "Fig_Correlation_Matrix_with_KDE")

### 2.4 Data visualisation — violin plots

In [ ]:
# (a) All variables on a common normalised scale
norm_df = (df[cols] - df[cols].min()) / (df[cols].max() - df[cols].min())
long = norm_df.melt(var_name="Variable", value_name="Normalised value")
long["Variable"] = long["Variable"].map(lambda c: LABELS.get(c, c))
fig, ax = plt.subplots(figsize=(11, 5))
sns.violinplot(data=long, x="Variable", y="Normalised value", hue="Variable", palette="Set2",
               inner="box", cut=0, linewidth=1, legend=False, ax=ax)
ax.set_xlabel("Variables"); ax.set_ylabel("Normalised value [0–1]")
save_fig(fig, "eda", "Fig_Violin_All_Normalised")

# (b) Individual violins in original units with mean marker — 2 figures per row
n = len(cols); ncol = 2; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 3.2 * nrow))
pal = sns.color_palette("Set2", n)
for i, (c, ax) in enumerate(zip(cols, axes.flat)):
    sns.violinplot(x=df[c], color=pal[i], inner="quartile", cut=0, linewidth=1, ax=ax)
    sns.stripplot(x=df[c], color="0.25", size=1.6, alpha=0.35, jitter=0.18, ax=ax)
    ax.scatter(df[c].mean(), 0, marker="D", color="red", s=28, zorder=5, label="Mean")
    ax.set_xlabel(LABELS.get(c, c)); ax.set_yticks([])
    ax.set_title(f"({chr(97+i)})", loc="left", fontsize=12)
for ax in axes.flat[n:]:
    ax.axis("off")
axes.flat[0].legend(loc="upper right", fontsize=8)
fig.tight_layout()
save_fig(fig, "eda", "Fig_Violin_Individual")

# (c) Histogram + KDE of each variable — 2 figures per row
fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 3.0 * nrow))
for i, (c, ax) in enumerate(zip(cols, axes.flat)):
    sns.histplot(df[c], kde=True, color=pal[i], edgecolor="white", ax=ax)
    ax.set_xlabel(LABELS.get(c, c)); ax.set_ylabel("Frequency" if i % ncol == 0 else "")
    ax.set_title(f"({chr(97+i)})", loc="left", fontsize=12)
for ax in axes.flat[n:]:
    ax.axis("off")
fig.tight_layout()
save_fig(fig, "eda", "Fig_Histogram_KDE")

## 3. Train/test split & K-fold cross-validation scheme

In [ ]:
X = df[FEATURES].values.astype(float)
y = df[TARGET].values.astype(float)
groups = df["Pile_ID"].values
idx = np.arange(len(df))

if SPLIT_MODE == "group":
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
    tr_idx, te_idx = next(gss.split(X, y, groups))
else:
    tr_idx, te_idx = train_test_split(idx, test_size=TEST_SIZE, random_state=SEED, shuffle=True)

X_train, X_test, y_train, y_test = X[tr_idx], X[te_idx], y[tr_idx], y[te_idx]
g_train = groups[tr_idx]
print(f"Split mode: {SPLIT_MODE} | Train: {len(tr_idx)} | Test: {len(te_idx)}")

# Fixed CV folds on the training set (same folds for every model & optimiser)
if SPLIT_MODE == "group":
    CV_SPLITS = list(GroupKFold(n_splits=K_FOLDS).split(X_train, y_train, g_train))
else:
    CV_SPLITS = list(KFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED).split(X_train))

train_df = df.iloc[tr_idx].copy(); train_df["Set"] = "Train"
test_df  = df.iloc[te_idx].copy();  test_df["Set"] = "Test"
with pd.ExcelWriter(f"{DIRS['data']}/Train_Test_Split.xlsx") as w:
    train_df.to_excel(w, sheet_name="Train", index=False)
    test_df.to_excel(w, sheet_name="Test", index=False)
    pd.concat([train_df[cols].describe().T.add_prefix("Train_"),
               test_df[cols].describe().T.add_prefix("Test_")], axis=1).to_excel(w, sheet_name="Statistics")

# Train vs test distribution check — 2 figures per row
n = len(cols); ncol = 2; nrow = int(np.ceil(n / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(5.0 * ncol, 3.0 * nrow))
for i, (c, ax) in enumerate(zip(cols, axes.flat)):
    sns.kdeplot(train_df[c], fill=True, color="#4C72B0", alpha=0.4, label="Training", ax=ax)
    sns.kdeplot(test_df[c], fill=True, color="#C44E52", alpha=0.4, label="Testing", ax=ax)
    ax.set_xlabel(LABELS.get(c, c)); ax.set_ylabel("Density" if i % ncol == 0 else "")
    ax.set_title(f"({chr(97+i)})", loc="left", fontsize=12)
for ax in axes.flat[n:]:
    ax.axis("off")
axes.flat[0].legend(fontsize=8)
fig.tight_layout()
save_fig(fig, "eda", "Fig_Train_Test_Distribution")

# K-fold scheme diagram
fig, ax = plt.subplots(figsize=(8, 0.6 * K_FOLDS + 1))
blk = len(X_train) / K_FOLDS
for k, (a, b) in enumerate(CV_SPLITS):
    for j in range(K_FOLDS):
        val = j == k
        ax.barh(k, blk, left=j * blk, color="#DD8452" if val else "#4C72B0", edgecolor="white", lw=1.5)
        ax.text(j * blk + blk / 2, k, f"Val ({len(b)})" if val else "Train", ha="center", va="center",
                color="k" if val else "white", fontsize=9)
ax.barh(K_FOLDS, len(X_train), color="0.6"); ax.barh(K_FOLDS, len(X_test), left=len(X_train), color="#C44E52")
ax.text(len(X_train) / 2, K_FOLDS, "Training set", ha="center", va="center", color="white", fontsize=9)
ax.text(len(X_train) + len(X_test) / 2, K_FOLDS, "Test", ha="center", va="center", color="white", fontsize=9)
ax.set_yticks(range(K_FOLDS + 1)); ax.set_yticklabels([f"Fold {k+1}" for k in range(K_FOLDS)] + ["Whole data"])
ax.invert_yaxis(); ax.set_xlabel("Number of samples (validation folds shown schematically)")
ax.tick_params(right=False, top=False)
save_fig(fig, "cv", "Fig_KFold_Scheme")

## 4. LSSVM model (RBF kernel)
LSSVM solves the linear KKT system
$\begin{bmatrix}0 & \mathbf{1}^T\\ \mathbf{1} & \mathbf{K}+\gamma^{-1}\mathbf{I}\end{bmatrix}\begin{bmatrix}b\\ \boldsymbol{\alpha}\end{bmatrix}=\begin{bmatrix}0\\ \mathbf{y}\end{bmatrix}$,
with $K(x_i,x_j)=\exp(-\lVert x_i-x_j\rVert^2/2\sigma^2)$. Inputs and output are min-max scaled inside the pipeline (fitted on training folds only — no leakage).

In [ ]:
class LSSVM(BaseEstimator, RegressorMixin):
    """Least Squares Support Vector Machine regressor with RBF kernel."""
    def __init__(self, gamma=1.0, sigma=1.0):
        self.gamma = gamma
        self.sigma = sigma

    def _kernel(self, A, B):
        sq = (np.sum(A**2, 1)[:, None] + np.sum(B**2, 1)[None, :] - 2 * A @ B.T)
        return np.exp(-np.maximum(sq, 0) / (2 * self.sigma**2))

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y, float).ravel()
        n = len(y)
        A = np.zeros((n + 1, n + 1))
        A[0, 1:] = 1.0; A[1:, 0] = 1.0
        A[1:, 1:] = self._kernel(X, X) + np.eye(n) / self.gamma
        rhs = np.concatenate([[0.0], y])
        try:
            sol = solve(A, rhs, assume_a="sym")
        except Exception:
            sol = np.linalg.lstsq(A, rhs, rcond=None)[0]
        self.b_, self.alpha_, self.X_fit_ = sol[0], sol[1:], X
        return self

    def predict(self, X):
        return self._kernel(np.asarray(X, float), self.X_fit_) @ self.alpha_ + self.b_


def build_model(gamma, sigma):
    """MinMax(X) -> LSSVM, with MinMax(y) target transform."""
    return TransformedTargetRegressor(
        regressor=Pipeline([("scaler", MinMaxScaler()), ("lssvm", LSSVM(gamma=gamma, sigma=sigma))]),
        transformer=MinMaxScaler())

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

def cv_rmse(gamma, sigma):
    """Mean K-fold CV RMSE on the training set (fitness function)."""
    errs = []
    for a, b in CV_SPLITS:
        m = build_model(gamma, sigma).fit(X_train[a], y_train[a])
        errs.append(rmse(y_train[b], m.predict(X_train[b])))
    return float(np.mean(errs))

t0 = time.time()
print("Default LSSVM CV-RMSE:", round(cv_rmse(**DEFAULT_LSSVM), 4), f"({time.time()-t0:.2f} s)")

## 5. Hyperparameter optimisation (PSO, BO, GWO)
Fitness = mean 5-fold CV-RMSE on the training set. Search in log10(γ), log10(σ) with identical bounds and iterations for all optimisers.
* **PSO** and **GWO**: population of `POP_SIZE` agents, `MAX_ITER` iterations.
* **BO**: Gaussian-process surrogate (Matérn 5/2) with expected-improvement acquisition; `POP_SIZE` Latin-hypercube initial points, then one new evaluation per iteration (far fewer function evaluations by design).

In [ ]:
LB = np.array([LOG_BOUNDS["gamma"][0], LOG_BOUNDS["sigma"][0]])
UB = np.array([LOG_BOUNDS["gamma"][1], LOG_BOUNDS["sigma"][1]])
DIM = 2

class Objective:
    """Wraps the fitness function and records every evaluation."""
    def __init__(self):
        self.history = []
    def __call__(self, z):
        z = np.clip(z, LB, UB)
        f = cv_rmse(10 ** z[0], 10 ** z[1])
        self.history.append((z[0], z[1], f))
        return f

def _result(name, best_x, best_f, curve, mean_curve, best_pos, obj, t):
    return {"name": name, "best_x": np.asarray(best_x), "best_f": float(best_f), "curve": np.array(curve),
            "mean_curve": np.array(mean_curve), "best_pos": np.array(best_pos),
            "history": np.array(obj.history), "n_eval": len(obj.history), "time": t}

# ---------------- Particle Swarm Optimisation ----------------
def PSO(obj, pop=POP_SIZE, iters=MAX_ITER, seed=SEED, w_max=0.9, w_min=0.4, c1=2.0, c2=2.0):
    rng = np.random.default_rng(seed); t0 = time.time()
    X = LB + rng.random((pop, DIM)) * (UB - LB)
    vmax = 0.2 * (UB - LB); V = rng.uniform(-vmax, vmax, (pop, DIM))
    fit = np.array([obj(x) for x in X])
    pbest, pfit = X.copy(), fit.copy()
    g = np.argmin(pfit); gbest, gfit = pbest[g].copy(), pfit[g]
    curve, mean_curve, best_pos = [], [], []
    for t in range(iters):
        w = w_max - (w_max - w_min) * t / iters
        r1, r2 = rng.random((pop, DIM)), rng.random((pop, DIM))
        V = np.clip(w * V + c1 * r1 * (pbest - X) + c2 * r2 * (gbest - X), -vmax, vmax)
        X = np.clip(X + V, LB, UB)
        fit = np.array([obj(x) for x in X])
        imp = fit < pfit
        pbest[imp], pfit[imp] = X[imp], fit[imp]
        g = np.argmin(pfit)
        if pfit[g] < gfit:
            gbest, gfit = pbest[g].copy(), pfit[g]
        curve.append(gfit); mean_curve.append(fit.mean()); best_pos.append(gbest.copy())
    return _result("PSO", gbest, gfit, curve, mean_curve, best_pos, obj, time.time() - t0)

# ---------------- Bayesian Optimisation (Gaussian process + expected improvement) ----------------
def BO(obj, n_init=POP_SIZE, iters=MAX_ITER, seed=SEED, xi=0.01, n_cand=4000):
    """GP surrogate (Matérn 5/2) on log(CV-RMSE); one new evaluation per iteration chosen by maximising EI."""
    rng = np.random.default_rng(seed); t0 = time.time()
    to_x = lambda u: LB + np.asarray(u) * (UB - LB)                 # unit cube -> log10 space
    U = qmc.LatinHypercube(d=DIM, seed=seed).random(n_init)        # space-filling initial design
    Y = np.array([obj(to_x(u)) for u in U])
    kernel = (ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=[0.3] * DIM, length_scale_bounds=(1e-2, 10.0), nu=2.5)
              + WhiteKernel(1e-4, (1e-8, 1e-1)))
    curve, mean_curve, best_pos = [], [], []
    for t in range(iters):
        gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=2, random_state=seed)
        gp.fit(U, np.log(Y))
        f_best = np.log(Y.min())

        def ei(u):
            u = np.atleast_2d(u); mu, sd = gp.predict(u, return_std=True); sd = np.maximum(sd, 1e-9)
            imp = f_best - mu - xi; z = imp / sd
            return imp * normal_dist.cdf(z) + sd * normal_dist.pdf(z)

        cand = rng.random((n_cand, DIM)); e = ei(cand)                # global random screening
        u_next, e_next = cand[np.argmax(e)], e.max()
        for s in cand[np.argsort(e)[-5:]]:                           # local refinement of the best 5
            r = minimize(lambda u: -ei(u)[0], s, bounds=[(0, 1)] * DIM, method="L-BFGS-B")
            if r.success and -r.fun > e_next:
                u_next, e_next = r.x, -r.fun
        if np.min(np.linalg.norm(U - u_next, axis=1)) < 1e-6:        # avoid re-sampling the same point
            u_next = rng.random(DIM)
        y_next = obj(to_x(u_next))
        U = np.vstack([U, u_next]); Y = np.append(Y, y_next)
        b = np.argmin(Y)
        curve.append(Y[b]); mean_curve.append(y_next); best_pos.append(to_x(U[b]))
    b = np.argmin(Y)
    return _result("BO", to_x(U[b]), Y[b], curve, mean_curve, best_pos, obj, time.time() - t0)

# ---------------- Grey Wolf Optimiser ----------------
def GWO(obj, pop=POP_SIZE, iters=MAX_ITER, seed=SEED):
    rng = np.random.default_rng(seed); t0 = time.time()
    X = LB + rng.random((pop, DIM)) * (UB - LB)
    fit = np.array([obj(x) for x in X])
    o = np.argsort(fit)
    leaders = [X[o[k]].copy() for k in range(3)]; lfit = [fit[o[k]] for k in range(3)]
    curve, mean_curve, best_pos = [], [], []
    for t in range(iters):
        a = 2 - 2 * t / iters
        for i in range(pop):
            new = np.zeros(DIM)
            for L in leaders:
                A = 2 * a * rng.random(DIM) - a; C = 2 * rng.random(DIM)
                new += L - A * np.abs(C * L - X[i])
            X[i] = np.clip(new / 3, LB, UB); fit[i] = obj(X[i])
        allX = np.vstack([X] + leaders); allf = np.concatenate([fit, lfit])
        o = np.argsort(allf)
        leaders = [allX[o[k]].copy() for k in range(3)]; lfit = [allf[o[k]] for k in range(3)]
        curve.append(lfit[0]); mean_curve.append(fit.mean()); best_pos.append(leaders[0].copy())
    return _result("GWO", leaders[0], lfit[0], curve, mean_curve, best_pos, obj, time.time() - t0)

ALGOS = {"PSO": PSO, "BO": BO, "GWO": GWO}
# label of the second curve in the convergence plots (BO has no population)
MEAN_LABEL = {"PSO": "Population mean", "BO": "Evaluated point", "GWO": "Population mean"}

In [ ]:
OPT_RESULTS = {}
for name in OPTIMISERS:
    res = ALGOS[name](Objective())
    OPT_RESULTS[name] = res
    print(f"{name}: best CV-RMSE = {res['best_f']:.4f} | gamma = {10**res['best_x'][0]:.4g} | "
          f"sigma = {10**res['best_x'][1]:.4g} | evaluations = {res['n_eval']} | time = {res['time']:.1f} s")
    if res["best_x"][0] >= LB[0] + 0.995 * (UB[0] - LB[0]):
        print(f"   note: {name} optimum lies on the upper gamma bound (10^{UB[0]:g})")

### 5.1 Convergence curves, hyperparameter evolution, search history

In [ ]:
it = np.arange(1, MAX_ITER + 1)
mk_every = max(1, MAX_ITER // 10)

# Save convergence data
conv = pd.DataFrame({"Iteration": it})
for k, r in OPT_RESULTS.items():
    conv[f"{k}_Best_CV_RMSE"] = r["curve"]; conv[f"{k}_Mean_Pop_RMSE"] = r["mean_curve"]
    conv[f"{k}_gamma"] = 10 ** r["best_pos"][:, 0]; conv[f"{k}_sigma"] = 10 ** r["best_pos"][:, 1]
with pd.ExcelWriter(f"{DIRS['hyper']}/Convergence_History.xlsx") as w:
    conv.to_excel(w, sheet_name="Convergence", index=False)
    for k, r in OPT_RESULTS.items():
        pd.DataFrame({"log10_gamma": r["history"][:, 0], "log10_sigma": r["history"][:, 1],
                      "CV_RMSE": r["history"][:, 2]}).to_excel(w, sheet_name=f"{k}_all_evaluations", index=False)

# (a) Combined convergence curve  +  (b) fitness box plot — 2 figures in one row
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for k, r in OPT_RESULTS.items():
    ax.plot(it, r["curve"], color=OPT_COLORS[k], marker=MODEL_MARKERS[f"LSSVM-{k}"], markevery=mk_every,
            ms=7, mec="k", mew=0.6, label=f"LSSVM-{k} (best = {r['best_f']:.4f})")
ax.set_xlabel("Iteration"); ax.set_ylabel(f"Best fitness ({K_FOLDS}-fold CV RMSE)")
ax.set_xlim(1, MAX_ITER); ax.legend(loc="upper right"); style_ax(ax)
ax.set_title("(a) Convergence curves", loc="left")

ax = axes[1]
bp = pd.DataFrame([(f"LSSVM-{k}", f) for k, r in OPT_RESULTS.items() for f in r["history"][:, 2]],
                  columns=["Model", "CV RMSE"])
sns.boxplot(data=bp, x="Model", y="CV RMSE", hue="Model", palette=MODEL_COLORS, showfliers=False, width=0.55,
            linewidth=1.2, legend=False, ax=ax,
            boxprops={"edgecolor": "k"}, medianprops={"color": "k", "lw": 1.6},
            whiskerprops={"color": "k"}, capprops={"color": "k"})
ax.set_yscale("log"); ax.set_xlabel(""); ax.set_ylabel(f"{K_FOLDS}-fold CV RMSE (all evaluations)")
style_ax(ax, logy=True); ax.xaxis.set_minor_locator(plt.NullLocator())
ax.set_title("(b) Fitness distribution", loc="left")
fig.tight_layout()
save_fig(fig, "opt", "Fig_Convergence_and_Fitness_Boxplot")

# (c) Individual: best vs population mean (2 x 2 with a comparison panel when 3 optimisers, else 1 row)
COMP = len(OPT_RESULTS) > 2
fig, axes = plt.subplots(2, 2, figsize=(13, 10)) if COMP else plt.subplots(1, len(OPT_RESULTS), figsize=(13, 5.2))
axes = np.atleast_1d(axes)
for ax, (k, r), lab_ in zip(axes.flat, OPT_RESULTS.items(), "abc"):
    ax.plot(it, r["mean_curve"], color="0.45", lw=1.0 if k == "BO" else 1.4, ls="--", label=MEAN_LABEL[k])
    ax.plot(it, r["curve"], color=OPT_COLORS[k], lw=2.4, marker=MODEL_MARKERS[f"LSSVM-{k}"],
            markevery=mk_every, ms=7, mec="k", mew=0.6, label="Best so far")
    ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_ylabel("CV RMSE (log scale)")
    ax.set_xlim(1, MAX_ITER); ax.set_title(f"({lab_}) LSSVM-{k}", loc="left"); ax.legend(loc="upper right")
    style_ax(ax, logy=True)
if COMP:
    ax = axes.flat[3]
    for k, r in OPT_RESULTS.items():
        ax.plot(it, r["curve"], color=OPT_COLORS[k], marker=MODEL_MARKERS[f"LSSVM-{k}"], markevery=mk_every,
                ms=7, mec="k", mew=0.6, label=f"LSSVM-{k}")
    ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_ylabel("Best CV RMSE (log scale)")
    ax.set_xlim(1, MAX_ITER); ax.set_title("(d) Comparison of optimisers", loc="left"); ax.legend(loc="upper right")
    style_ax(ax, logy=True)
fig.tight_layout()
save_fig(fig, "opt", "Fig_Convergence_Best_vs_Mean")

# (e) Evolution of hyperparameters — 2 figures in one row
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for k, r in OPT_RESULTS.items():
    for ax, col in zip(axes, [0, 1]):
        ax.plot(it, 10 ** r["best_pos"][:, col], color=OPT_COLORS[k], marker=MODEL_MARKERS[f"LSSVM-{k}"],
                markevery=mk_every, ms=7, mec="k", mew=0.6, drawstyle="steps-post", label=f"LSSVM-{k}")
axes[0].set_ylabel("Regularisation parameter, $\\gamma$"); axes[1].set_ylabel("Kernel width, $\\sigma$")
for ax, l, t in zip(axes, "ab", ["Regularisation parameter", "Kernel width"]):
    ax.set_yscale("log"); ax.set_xlabel("Iteration"); ax.set_xlim(1, MAX_ITER); ax.legend(loc="best")
    ax.set_title(f"({l}) {t}", loc="left"); style_ax(ax, logy=True)
fig.tight_layout()
save_fig(fig, "opt", "Fig_Hyperparameter_Evolution")

In [ ]:
# (e) Fitness landscape + search history of each optimiser — 2 figures per row
if PLOT_LANDSCAPE:
    from matplotlib.ticker import AutoMinorLocator
    from matplotlib.colors import BoundaryNorm
    from scipy.ndimage import zoom

    # --- evaluate CV-RMSE on a grid and save ---
    gg = np.linspace(LB[0], UB[0], LANDSCAPE_GRID); ss = np.linspace(LB[1], UB[1], LANDSCAPE_GRID)
    Z = np.array([[cv_rmse(10 ** a, 10 ** b) for a in gg] for b in ss])
    pd.DataFrame(Z, index=np.round(ss, 3), columns=np.round(gg, 3)).to_excel(
        f"{DIRS['hyper']}/Fitness_Landscape_log10sigma_x_log10gamma.xlsx")

    # --- smooth the surface for plotting (cubic interpolation, ×4 resolution) ---
    up = 4
    ZL = np.log10(Z)
    ZL_s = zoom(ZL, up, order=3)
    gg_s = np.linspace(LB[0], UB[0], ZL_s.shape[1]); ss_s = np.linspace(LB[1], UB[1], ZL_s.shape[0])
    levels = np.linspace(ZL_s.min(), ZL_s.max(), 30)
    norm_ = BoundaryNorm(levels, ncolors=256)
    CMAP = "viridis_r"

    def landscape(ax):
        cf = ax.contourf(gg_s, ss_s, ZL_s, levels=levels, cmap=CMAP, norm=norm_)
        cl = ax.contour(gg_s, ss_s, ZL_s, levels=levels[::4], colors="k", linewidths=0.5, alpha=0.55)
        ax.clabel(cl, fmt=lambda v: f"{10**v:.2f}", fontsize=8, inline=True)
        ax.set_xlim(LB[0], UB[0]); ax.set_ylim(LB[1], UB[1])
        ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.tick_params(which="both", direction="out", top=False, right=False)
        ax.set_xlabel("log$_{10}$($\\gamma$)"); ax.set_ylabel("log$_{10}$($\\sigma$)")
        return cf

    COMP = len(OPT_RESULTS) > 2
    fig, axes = plt.subplots(2, 2, figsize=(13, 10.5)) if COMP else plt.subplots(1, len(OPT_RESULTS), figsize=(13, 5.6))
    axes = np.atleast_1d(axes)
    for ax, (k, r), l in zip(axes.flat, OPT_RESULTS.items(), "abc"):
        cf = landscape(ax)
        h = r["history"]; bp_ = r["best_pos"]
        ax.scatter(h[:, 0], h[:, 1], s=10, c="white", edgecolor="0.25", lw=0.3, alpha=0.7,
                   zorder=3, label="Evaluated solutions")
        ax.plot(bp_[:, 0], bp_[:, 1], "-", color=OPT_COLORS[k], lw=2.2, zorder=4,
                path_effects=[matplotlib.patheffects.withStroke(linewidth=3.6, foreground="white")],
                label="Best-so-far path")
        ax.scatter(*bp_[0], marker="o", s=70, c="white", edgecolor="k", lw=1.2, zorder=5, label="Start")
        ax.scatter(*r["best_x"], marker="*", s=380, c="red", edgecolor="k", lw=1.0, zorder=6, label="Optimum")
        ax.annotate(f"$\\gamma$ = {10**r['best_x'][0]:.3g}\n$\\sigma$ = {10**r['best_x'][1]:.3g}\n"
                    f"RMSE = {r['best_f']:.4f}",
                    xy=(0.04, 0.96), xycoords="axes fraction", va="top", fontsize=10,
                    bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.3", alpha=0.95), zorder=7)
        ax.set_title(f"({l}) LSSVM-{k}", loc="left")
    axes.flat[0].legend(loc="lower right", fontsize=9, framealpha=0.95)

    if COMP:   # (d) all optima on one landscape
        ax = axes[1, 1]
        cf = landscape(ax)
        for k, r in OPT_RESULTS.items():
            ax.plot(r["best_pos"][:, 0], r["best_pos"][:, 1], "-", color=OPT_COLORS[k], lw=2, zorder=4,
                    path_effects=[matplotlib.patheffects.withStroke(linewidth=3.4, foreground="white")])
            ax.scatter(*r["best_x"], marker=MODEL_MARKERS[f"LSSVM-{k}"], s=150, c=OPT_COLORS[k], edgecolor="k",
                       lw=1.2, zorder=6, label=f"LSSVM-{k} (RMSE = {r['best_f']:.4f})")
        ax.set_title("(d) Comparison of optima", loc="left"); ax.legend(loc="lower right", fontsize=9)

    fig.tight_layout(rect=[0, 0, 0.9, 1])
    cax = fig.add_axes([0.915, 0.12, 0.018, 0.76])
    cb = fig.colorbar(cf, cax=cax)
    ticks = np.linspace(levels[0], levels[-1], 7)
    cb.set_ticks(ticks); cb.set_ticklabels([f"{10**t:.2f}" for t in ticks])
    cb.set_label(f"{K_FOLDS}-fold CV RMSE (log colour scale)"); cb.ax.tick_params(direction="out")
    save_fig(fig, "opt", "Fig_Fitness_Landscape_Search_History")

    # (f) 3-D surface of the fitness landscape — 2 views in one row
    fig = plt.figure(figsize=(14, 6))
    GX, GY = np.meshgrid(gg_s, ss_s)
    for i, (elev, azim) in enumerate([(30, -130), (55, -60)]):
        ax = fig.add_subplot(1, 2, i + 1, projection="3d")
        surf = ax.plot_surface(GX, GY, ZL_s, cmap=CMAP, norm=norm_, linewidth=0, antialiased=True,
                               rstride=1, cstride=1, alpha=0.95)
        ax.contour(GX, GY, ZL_s, levels=levels[::3], zdir="z", offset=ZL_s.min() - 0.15, cmap=CMAP,
                   norm=norm_, linewidths=0.8)
        z0 = ZL_s.min() - 0.15                      # floor level for projected contours
        for k, r in OPT_RESULTS.items():           # optima shown on the floor + vertical drop line
            bx, by, bz = r["best_x"][0], r["best_x"][1], np.log10(r["best_f"])
            ax.plot([bx, bx], [by, by], [z0, bz], color=OPT_COLORS[k], lw=1.6, ls="--", zorder=9)
            ax.scatter(bx, by, z0, marker=MODEL_MARKERS[f"LSSVM-{k}"], s=110, c=OPT_COLORS[k],
                       edgecolor="k", depthshade=False, zorder=10, label=f"LSSVM-{k}")
        ax.set_zlim(ZL_s.min() - 0.15, ZL_s.max())
        ax.set_xlabel("log$_{10}$($\\gamma$)", labelpad=8); ax.set_ylabel("log$_{10}$($\\sigma$)", labelpad=8)
        ax.set_zlabel("log$_{10}$(CV RMSE)", labelpad=8)
        ax.view_init(elev=elev, azim=azim)
        ax.xaxis.pane.set_facecolor("white"); ax.yaxis.pane.set_facecolor("white"); ax.zaxis.pane.set_facecolor("white")
        ax.set_title(f"({'ab'[i]})", loc="left")
        if i == 0:
            ax.legend(loc="upper left", fontsize=9)
    cb3 = fig.colorbar(surf, ax=fig.axes, shrink=0.6, pad=0.06)
    t3 = np.linspace(levels[0], levels[-1], 7)
    cb3.set_ticks(t3); cb3.set_ticklabels([f"{10**t:.2f}" for t in t3])
    cb3.set_label(f"{K_FOLDS}-fold CV RMSE (log colour scale)")
    save_fig(fig, "opt", "Fig_Fitness_Landscape_3D")

### 5.2 Error curve — RMSE of the best-so-far solution vs iteration
For each iteration, the best hyperparameters are re-trained on the full training set; training and testing RMSE are tracked (test set used for **monitoring only**, never for selection).

In [ ]:
# Error curve — RMSE of the best-so-far solution vs iteration (2 figures per row)
# Test set is used for MONITORING only (never for selecting hyperparameters)
import matplotlib.patheffects
mk_every = max(1, MAX_ITER // 10)
ERR_COLORS = {"Training": "#1F4E9E", "CV": "#E08214", "Testing": "#B2182B"}

err_curve = pd.DataFrame({"Iteration": it})
ERR = {}
for k, r in OPT_RESULTS.items():
    cache, tr_e, te_e, tr_r2, te_r2 = {}, [], [], [], []
    for p in r["best_pos"]:
        key = tuple(np.round(p, 10))
        if key not in cache:
            m = build_model(10 ** p[0], 10 ** p[1]).fit(X_train, y_train)
            ptr, pte = m.predict(X_train), m.predict(X_test)
            cache[key] = (rmse(y_train, ptr), rmse(y_test, pte), r2_score(y_train, ptr), r2_score(y_test, pte))
        a, b, c, d = cache[key]
        tr_e.append(a); te_e.append(b); tr_r2.append(c); te_r2.append(d)
    ERR[k] = {"train": np.array(tr_e), "cv": np.asarray(r["curve"]), "test": np.array(te_e)}
    err_curve[f"{k}_Train_RMSE"] = tr_e; err_curve[f"{k}_CV_RMSE"] = r["curve"]
    err_curve[f"{k}_Test_RMSE"] = te_e; err_curve[f"{k}_Train_R2"] = tr_r2; err_curve[f"{k}_Test_R2"] = te_r2
err_curve.to_excel(f"{DIRS['hyper']}/Error_Curve_vs_Iteration.xlsx", index=False)

ymax = max(max(e["train"].max(), e["cv"].max(), e["test"].max()) for e in ERR.values()) * 1.08
ymin = min(min(e["train"].min(), e["cv"].min(), e["test"].min()) for e in ERR.values()) * 0.9

COMP = len(ERR) > 2
fig, axes = plt.subplots(2, 2, figsize=(13, 10)) if COMP else plt.subplots(1, len(ERR), figsize=(13, 5.2))
axes = np.atleast_1d(axes)
for ax, (k, e), l in zip(axes.flat, ERR.items(), "abc"):
    ax.fill_between(it, e["train"], e["test"], color="0.85", alpha=0.7, lw=0, label="Generalisation gap")
    ax.plot(it, e["train"], color=ERR_COLORS["Training"], lw=2.2, marker="o", markevery=mk_every, ms=6,
            mec="k", mew=0.5, label="Training")
    ax.plot(it, e["cv"], color=ERR_COLORS["CV"], lw=2.2, ls="--", marker="s", markevery=mk_every, ms=6,
            mec="k", mew=0.5, label=f"{K_FOLDS}-fold CV (fitness)")
    ax.plot(it, e["test"], color=ERR_COLORS["Testing"], lw=2.2, ls="-.", marker="^", markevery=mk_every, ms=7,
            mec="k", mew=0.5, label="Testing")
    # final values
    txt = (f"Final RMSE\nTrain = {e['train'][-1]:.4f}\nCV = {e['cv'][-1]:.4f}\nTest = {e['test'][-1]:.4f}")
    ax.text(0.97, 0.60, txt, transform=ax.transAxes, ha="right", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.3", alpha=0.95))
    ax.set_xlim(1, MAX_ITER); ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Iteration"); ax.set_ylabel("RMSE")
    ax.set_title(f"({l}) LSSVM-{k}", loc="left"); ax.legend(loc="upper right", fontsize=9.5)
    style_ax(ax)

if COMP:   # (d) Testing-error comparison of all optimisers
    ax = axes[1, 1]
    for k, e in ERR.items():
        ax.plot(it, e["test"], color=OPT_COLORS[k], lw=2.2, marker=MODEL_MARKERS[f"LSSVM-{k}"], markevery=mk_every,
                ms=7, mec="k", mew=0.6, label=f"LSSVM-{k} (final = {e['test'][-1]:.4f})")
        ax.plot(it, e["train"], color=OPT_COLORS[k], lw=1.2, ls=":", alpha=0.9)
    ax.plot([], [], color="0.3", lw=2.2, label="Testing (solid)")
    ax.plot([], [], color="0.3", lw=1.2, ls=":", label="Training (dotted)")
    ax.set_xlim(1, MAX_ITER); ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Iteration"); ax.set_ylabel("RMSE")
    ax.set_title("(d) Comparison of optimisers", loc="left"); ax.legend(loc="upper right", fontsize=9.5)
    style_ax(ax)

fig.tight_layout()
save_fig(fig, "opt", "Fig_Error_Curve_vs_Iteration")

## 6. Final training — save hyperparameters & trained models

In [ ]:
HYPER = {"LSSVM": dict(DEFAULT_LSSVM)}
for k, r in OPT_RESULTS.items():
    HYPER[f"LSSVM-{k}"] = {"gamma": float(10 ** r["best_x"][0]), "sigma": float(10 ** r["best_x"][1])}

ALGO_PARAMS = {"PSO": "w: 0.9→0.4 (linear), c1 = c2 = 2.0, Vmax = 0.2·range",
               "BO":  f"GP surrogate (Matérn ν = 2.5) on log(CV-RMSE), EI acquisition (ξ = 0.01), {POP_SIZE} Latin-hypercube initial points",
               "GWO": "a: 2→0 (linear), α-β-δ leaders"}

MODELS, hp_rows = {}, []
for name, hp in HYPER.items():
    t0 = time.time()
    m = build_model(hp["gamma"], hp["sigma"]).fit(X_train, y_train)
    MODELS[name] = m
    joblib.dump(m, f"{DIRS['models']}/{name}.joblib")
    core = m.regressor_.named_steps["lssvm"]
    np.savez(f"{DIRS['models']}/{name}_weights.npz", alpha=core.alpha_, b=core.b_, X_support_scaled=core.X_fit_)
    opt = name.split("-")[1] if "-" in name else None
    row = {"Model": name, "gamma (regularisation)": hp["gamma"], "sigma (RBF width)": hp["sigma"],
           "sigma^2": hp["sigma"] ** 2, "Kernel": "RBF", "Bias b (scaled)": float(core.b_),
           f"{K_FOLDS}-fold CV RMSE": cv_rmse(hp["gamma"], hp["sigma"]),
           "Training time (s)": time.time() - t0}
    if opt:
        r = OPT_RESULTS[opt]
        row.update({"Optimiser": opt,
                    "Population / initial points": POP_SIZE, "Max iterations": MAX_ITER,
                    "Function evaluations": r["n_eval"], "Optimisation time (s)": r["time"],
                    "log10(gamma) bounds": str(LOG_BOUNDS["gamma"]), "log10(sigma) bounds": str(LOG_BOUNDS["sigma"]),
                    "Algorithm parameters": ALGO_PARAMS[opt]})
    else:
        row.update({"Optimiser": "None (default)"})
    hp_rows.append(row)

HP_TABLE = pd.DataFrame(hp_rows)
HP_TABLE.to_excel(f"{DIRS['hyper']}/Optimal_Hyperparameters.xlsx", index=False)
with open(f"{DIRS['hyper']}/Optimal_Hyperparameters.json", "w") as f:
    json.dump({"hyperparameters": HYPER, "config": {"SPLIT_MODE": SPLIT_MODE, "TEST_SIZE": TEST_SIZE,
               "K_FOLDS": K_FOLDS, "SEED": SEED, "POP_SIZE": POP_SIZE, "MAX_ITER": MAX_ITER,
               "LOG_BOUNDS": LOG_BOUNDS, "FEATURES": FEATURES, "TARGET": TARGET}}, f, indent=2)
HP_TABLE

## 7. K-fold cross-validation of final models & learning (error) curves

In [ ]:
cv_rows = []
for name, hp in HYPER.items():
    cvr = cross_validate(build_model(hp["gamma"], hp["sigma"]), X_train, y_train, cv=CV_SPLITS,
                         scoring={"R2": "r2", "RMSE": "neg_root_mean_squared_error",
                                  "MAE": "neg_mean_absolute_error"}, return_train_score=True)
    for k in range(K_FOLDS):
        cv_rows.append({"Model": name, "Fold": k + 1,
                        "Train R2": cvr["train_R2"][k], "Val R2": cvr["test_R2"][k],
                        "Train RMSE": -cvr["train_RMSE"][k], "Val RMSE": -cvr["test_RMSE"][k],
                        "Train MAE": -cvr["train_MAE"][k], "Val MAE": -cvr["test_MAE"][k]})
CV_DF = pd.DataFrame(cv_rows)
CV_SUM = CV_DF.groupby("Model", sort=False).agg(["mean", "std"]).drop(columns="Fold")
with pd.ExcelWriter(f"{DIRS['metric']}/KFold_CV_Results.xlsx") as w:
    CV_DF.to_excel(w, sheet_name="Fold-wise", index=False)
    CV_SUM.to_excel(w, sheet_name="Mean_Std")

# ---------------- 3-D style bar helper ----------------
from matplotlib.patches import Rectangle, Polygon
from matplotlib.colors import to_rgb
from matplotlib.ticker import AutoMinorLocator

def _shade(c, f):
    """f < 1 darker, f > 1 lighter."""
    r = np.array(to_rgb(c))
    return tuple(np.clip(r * f, 0, 1)) if f <= 1 else tuple(r + (1 - r) * (f - 1))

def bar3d(ax, x, h, w, color, y0, dx, dy, label=None, fmt="{:.3f}", fs=8.5, err=None):
    """Draw one pseudo-3-D bar (front + top + side) with its value at the top."""
    ax.add_patch(Rectangle((x - w / 2, y0), w, h - y0, fc=color, ec="k", lw=0.6, zorder=3))
    ax.add_patch(Polygon([(x - w / 2, h), (x + w / 2, h), (x + w / 2 + dx, h + dy), (x - w / 2 + dx, h + dy)],
                         closed=True, fc=_shade(color, 1.35), ec="k", lw=0.6, zorder=3))
    ax.add_patch(Polygon([(x + w / 2, y0), (x + w / 2 + dx, y0 + dy), (x + w / 2 + dx, h + dy), (x + w / 2, h)],
                         closed=True, fc=_shade(color, 0.65), ec="k", lw=0.6, zorder=3))
    top = h + dy
    if err is not None:
        ax.errorbar(x + dx / 2, h + dy / 2, yerr=err, fmt="none", ecolor="k", elinewidth=1.1, capsize=4, zorder=4)
        top = h + dy / 2 + err
    ax.text(x + dx / 2, top + dy * 0.6, fmt.format(h), ha="center", va="bottom", rotation=90,
            fontsize=fs, zorder=5)

def grouped_bar3d(ax, groups, series, values, colors, ylim, errs=None, fmt="{:.3f}", fs=8.5):
    """groups: x-categories; series: bars within each group; values[s][g]."""
    n_s = len(series); gw = 0.8; w = gw / n_s * 0.85
    y0, y1 = ylim
    dx, dy = w * 0.35, (y1 - y0) * 0.025
    for gi, g in enumerate(groups):
        for si, s in enumerate(series):
            x = gi - gw / 2 + (si + 0.5) * gw / n_s
            bar3d(ax, x, values[s][gi], w, colors[s], y0, dx, dy, fmt=fmt, fs=fs,
                  err=None if errs is None else errs[s][gi])
    ax.set_xlim(-0.6, len(groups) - 0.4); ax.set_ylim(y0, y1)
    ax.set_xticks(range(len(groups))); ax.set_xticklabels(groups)
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(axis="x", which="both", top=False, bottom=False)
    ax.grid(True, axis="y", ls="--", lw=0.6, color="0.8", zorder=0); ax.set_axisbelow(True)
    handles = [Rectangle((0, 0), 1, 1, fc=colors[s], ec="k", lw=0.6) for s in series]
    ax.legend(handles, series, loc="upper center", ncol=len(series), fontsize=9.5,
              bbox_to_anchor=(0.5, -0.12), frameon=False)

def nice_ylim(vals, metric, errs=None):
    vals = np.asarray(vals, float); hi = vals.max() + (0 if errs is None else np.max(errs))
    if "R2" in metric:
        lo = max(0.0, np.floor((vals.min() - 0.05) * 20) / 20)
        return lo, lo + (hi - lo) * 1.30
    return 0.0, hi * 1.35

MODELS_ = list(HYPER.keys())
FOLDS_ = [f"Fold {k+1}" for k in range(K_FOLDS)]
PRETTY = {"Val R2": "Validation R$^2$", "Val RMSE": "Validation RMSE", "Val MAE": "Validation MAE"}

# ---- Figure 1: fold-wise validation metrics + mean ± std (2 × 2, 3-D bars) ----
fig, axes = plt.subplots(2, 2, figsize=(14, 10.5))
for ax, met, l in zip(axes.flat, ["Val R2", "Val RMSE", "Val MAE"], "abc"):
    vals = {m: CV_DF[CV_DF.Model == m].sort_values("Fold")[met].values for m in MODELS_}
    ylim = nice_ylim(np.concatenate(list(vals.values())), met)
    grouped_bar3d(ax, FOLDS_, MODELS_, vals, MODEL_COLORS, ylim, fmt="{:.3f}" if "R2" in met else "{:.3f}")
    ax.set_ylabel(PRETTY[met]); ax.set_title(f"({l}) {PRETTY[met]} per fold", loc="left")

ax = axes[1, 1]   # (d) mean ± std of validation RMSE / MAE per model
means = {m: [CV_SUM.loc[m, ("Val RMSE", "mean")], CV_SUM.loc[m, ("Val MAE", "mean")]] for m in MODELS_}
stds  = {m: [CV_SUM.loc[m, ("Val RMSE", "std")],  CV_SUM.loc[m, ("Val MAE", "std")]]  for m in MODELS_}
allv = np.concatenate([means[m] for m in MODELS_]); alle = np.concatenate([stds[m] for m in MODELS_])
grouped_bar3d(ax, ["Validation RMSE", "Validation MAE"], MODELS_, means, MODEL_COLORS,
              nice_ylim(allv, "RMSE", alle), errs=stds, fmt="{:.3f}")
ax.set_ylabel("Mean ± SD over folds"); ax.set_title(f"(d) Mean ± SD of {K_FOLDS}-fold CV errors", loc="left")
fig.tight_layout(h_pad=3)
save_fig(fig, "cv", "Fig_KFold_CV_Foldwise")

# ---- Figure 2: distribution across folds — 2 figures in one row ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, met, l in zip(axes, ["Val RMSE", "Val R2"], "ab"):
    sns.boxplot(data=CV_DF, x="Model", y=met, hue="Model", palette=MODEL_COLORS, width=0.5, legend=False,
                linewidth=1.2, showfliers=False, ax=ax, boxprops={"edgecolor": "k"},
                medianprops={"color": "k", "lw": 1.6}, whiskerprops={"color": "k"}, capprops={"color": "k"})
    sns.stripplot(data=CV_DF, x="Model", y=met, color="k", size=6, jitter=0.12, alpha=0.8,
                  edgecolor="white", linewidth=0.5, ax=ax)
    mu = CV_DF.groupby("Model", sort=False)[met].mean()
    for i, m in enumerate(MODELS_):
        ax.scatter(i, mu[m], marker="D", s=55, c="white", edgecolor="red", lw=1.5, zorder=6,
                   label="Mean" if i == 0 else None)
        ax.text(i + 0.28, mu[m], f"{mu[m]:.3f}", va="center", fontsize=9.5, color="red")
    ax.set_xlabel(""); ax.set_ylabel(f"{PRETTY[met]} ({K_FOLDS} folds)")
    ax.set_title(f"({l}) {PRETTY[met]}", loc="left"); ax.legend(loc="best")
    ax.yaxis.set_minor_locator(AutoMinorLocator()); ax.tick_params(axis="x", which="both", top=False)
    ax.grid(True, axis="y", ls="--", lw=0.6, color="0.8"); ax.set_axisbelow(True)
fig.tight_layout()
save_fig(fig, "cv", "Fig_KFold_CV_Boxplot")
CV_SUM.round(4)

In [ ]:
# Learning curves (training size vs error) — 2 figures per row
from matplotlib.ticker import AutoMinorLocator
sizes = np.linspace(0.1, 1.0, 10)
LC_COLORS = {"train": "#1F4E9E", "val": "#B2182B"}
lc_rows, LC = [], {}
for name, hp in HYPER.items():
    ns, tr, va = learning_curve(build_model(hp["gamma"], hp["sigma"]), X_train, y_train, cv=CV_SPLITS,
                                train_sizes=sizes, scoring="neg_root_mean_squared_error", shuffle=True,
                                random_state=SEED)
    tr, va = -tr, -va
    LC[name] = {"n": ns, "tr_m": tr.mean(1), "tr_s": tr.std(1), "va_m": va.mean(1), "va_s": va.std(1)}
    for s_, a, b, c, d in zip(ns, tr.mean(1), tr.std(1), va.mean(1), va.std(1)):
        lc_rows.append({"Model": name, "Train size": s_, "Train RMSE mean": a, "Train RMSE std": b,
                        "Val RMSE mean": c, "Val RMSE std": d, "Gap (Val - Train)": c - a})
pd.DataFrame(lc_rows).to_excel(f"{DIRS['metric']}/Learning_Curves.xlsx", index=False)

def _style(ax):
    ax.grid(True, ls="--", lw=0.6, color="0.8"); ax.set_axisbelow(True)
    ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())

ymax = max((d["va_m"] + d["va_s"]).max() for d in LC.values()) * 1.08
ymin = max(0, min((d["tr_m"] - d["tr_s"]).min() for d in LC.values()) * 0.9)

# ---- Figure 1: learning curve of each model (2 × 2) ----
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (name, d), l in zip(axes.flat, LC.items(), "abcd"):
    n_ = d["n"]
    ax.fill_between(n_, d["tr_m"], d["va_m"], color="0.88", lw=0, label="Generalisation gap")
    ax.fill_between(n_, d["tr_m"] - d["tr_s"], d["tr_m"] + d["tr_s"], color=LC_COLORS["train"], alpha=0.18, lw=0)
    ax.fill_between(n_, d["va_m"] - d["va_s"], d["va_m"] + d["va_s"], color=LC_COLORS["val"], alpha=0.18, lw=0)
    ax.plot(n_, d["tr_m"], "-o", color=LC_COLORS["train"], lw=2.2, ms=7, mec="k", mew=0.6, label="Training (mean ± SD)")
    ax.plot(n_, d["va_m"], "--s", color=LC_COLORS["val"], lw=2.2, ms=7, mec="k", mew=0.6,
            label=f"{K_FOLDS}-fold CV (mean ± SD)")
    ax.text(0.97, 0.55, f"At full size (n = {n_[-1]})\nTrain RMSE = {d['tr_m'][-1]:.4f}\n"
            f"CV RMSE = {d['va_m'][-1]:.4f}\nGap = {d['va_m'][-1] - d['tr_m'][-1]:.4f}",
            transform=ax.transAxes, ha="right", va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.3", alpha=0.95))
    ax.set_xlim(n_[0] * 0.9, n_[-1] * 1.02); ax.set_ylim(ymin, ymax)
    ax.set_xlabel("Number of training samples"); ax.set_ylabel("RMSE")
    ax.set_title(f"({l}) {name}", loc="left"); ax.legend(loc="upper right", fontsize=9.5)
    _style(ax)
fig.tight_layout()
save_fig(fig, "cv", "Fig_Learning_Curves")

# ---- Figure 2: comparison of models — 2 figures in one row ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, d in LC.items():
    axes[0].plot(d["n"], d["va_m"], "-", marker=MODEL_MARKERS[name], color=MODEL_COLORS[name], lw=2.2,
                 ms=7, mec="k", mew=0.6, label=name)
    axes[0].fill_between(d["n"], d["va_m"] - d["va_s"], d["va_m"] + d["va_s"], color=MODEL_COLORS[name],
                         alpha=0.12, lw=0)
    axes[1].plot(d["n"], d["va_m"] - d["tr_m"], "-", marker=MODEL_MARKERS[name], color=MODEL_COLORS[name],
                 lw=2.2, ms=7, mec="k", mew=0.6, label=name)
axes[0].set_ylabel(f"{K_FOLDS}-fold CV RMSE"); axes[0].set_title("(a) Validation error vs training size", loc="left")
axes[1].set_ylabel("Generalisation gap (CV − training RMSE)")
axes[1].set_title("(b) Generalisation gap vs training size", loc="left")
axes[1].axhline(0, color="k", lw=0.8)
for ax in axes:
    ax.set_xlabel("Number of training samples"); ax.legend(loc="upper right"); _style(ax)
fig.tight_layout()
save_fig(fig, "cv", "Fig_Learning_Curves_Comparison")

## 8. Model evaluation — 8 performance metrics & rank analysis

In [ ]:
# Eight performance metrics: R2, RMSE, MAE, MAPE, VAF, WI, a20-index, PI
N_FEAT = len(FEATURES)
METRIC_LIST = ["R2", "RMSE", "MAE", "MAPE (%)", "VAF (%)", "WI", "a20-index", "PI"]
LOWER = ["RMSE", "MAE", "MAPE (%)"]                       # lower is better
HIGHER = [m for m in METRIC_LIST if m not in LOWER]        # higher is better

def evaluate(yt, yp, k=N_FEAT):
    yt, yp = np.asarray(yt, float), np.asarray(yp, float); e = yt - yp; n = len(yt)
    nz = yt != 0                                            # MAPE and a20 use non-zero measured values
    R2 = r2_score(yt, yp); RMSE = rmse(yt, yp)
    VAF = (1 - np.var(e) / np.var(yt)) * 100
    adjR2 = 1 - (1 - R2) * (n - 1) / (n - k - 1)
    ratio = yp[nz] / yt[nz]
    return {
        "R2": R2, "RMSE": RMSE, "MAE": mean_absolute_error(yt, yp),
        "MAPE (%)": np.mean(np.abs(e[nz] / yt[nz])) * 100, "VAF (%)": VAF,
        "WI": 1 - np.sum(e**2) / np.sum((np.abs(yp - yt.mean()) + np.abs(yt - yt.mean()))**2),
        "a20-index": np.mean((ratio >= 0.8) & (ratio <= 1.2)),
        "PI": adjR2 + 0.01 * VAF - RMSE,
    }

PRED = {name: {"train": m.predict(X_train), "test": m.predict(X_test)} for name, m in MODELS.items()}
met_rows = []
for name in MODELS:
    for phase, key, yt in [("Training", "train", y_train), ("Testing", "test", y_test)]:
        met_rows.append({"Model": name, "Phase": phase, **evaluate(yt, PRED[name][key])})
METRICS = pd.DataFrame(met_rows)

# Rank-based score analysis (Zorlu et al. 2008): rank 1 (worst) ... n (best) per metric;
# values equal to 3 decimals get the same rank
score_tbl = []
for phase in ["Training", "Testing"]:
    sub = METRICS[METRICS.Phase == phase].set_index("Model")[METRIC_LIST].round(3)
    sc = pd.DataFrame(index=sub.index)
    for mtr in HIGHER: sc[mtr] = sub[mtr].rank(ascending=True, method="min").astype(int)
    for mtr in LOWER:  sc[mtr] = sub[mtr].rank(ascending=False, method="min").astype(int)
    sc = sc[METRIC_LIST]; sc["Score"] = sc.sum(axis=1); sc["Phase"] = phase
    score_tbl.append(sc.reset_index())
SCORE = pd.concat(score_tbl)
TOTAL = SCORE.pivot(index="Model", columns="Phase", values="Score")
TOTAL["Total score"] = TOTAL.sum(axis=1); TOTAL["Rank"] = TOTAL["Total score"].rank(ascending=False, method="min").astype(int)
TOTAL = TOTAL.sort_values("Rank")

# Best model = lowest 5-fold CV RMSE (the optimisation criterion; the test set is not used for selection)
BEST = HP_TABLE.loc[HP_TABLE[f"{K_FOLDS}-fold CV RMSE"].idxmin(), "Model"]
print("Best model (lowest 5-fold CV RMSE):", BEST)
print("Lowest testing RMSE:", METRICS[METRICS.Phase == "Testing"].sort_values("RMSE").iloc[0]["Model"])

with pd.ExcelWriter(f"{DIRS['metric']}/Performance_Metrics.xlsx") as w:
    METRICS.to_excel(w, sheet_name="Metrics", index=False)
    SCORE.to_excel(w, sheet_name="Rank_Scores", index=False)
    TOTAL.to_excel(w, sheet_name="Total_Score_Ranking")
METRICS.round(4)

In [ ]:
TOTAL

### 8.1 Save actual vs predicted values (Excel)

In [ ]:
with pd.ExcelWriter(f"{DIRS['pred']}/Actual_vs_Predicted_All_Models.xlsx") as w:
    for phase, ii, yt in [("Train", tr_idx, y_train), ("Test", te_idx, y_test)]:
        out = df.iloc[ii][["Pile_ID"] + FEATURES].reset_index().rename(columns={"index": "Row_in_raw_data"})
        out[f"Actual_{TARGET}"] = yt
        for name in MODELS:
            p = PRED[name][phase.lower()]
            out[f"{name}_Pred"] = p
            out[f"{name}_Residual"] = yt - p
            out[f"{name}_AbsError"] = np.abs(yt - p)
        out.to_excel(w, sheet_name=phase, index=False)
    full = df[["Pile_ID"] + FEATURES + [TARGET]].copy()
    full["Set"] = np.where(np.isin(idx, tr_idx), "Train", "Test")
    for name, m in MODELS.items():
        full[f"{name}_Pred"] = m.predict(X)
    full.to_excel(w, sheet_name="Whole_dataset", index=False)
# one file per model too
for name in MODELS:
    with pd.ExcelWriter(f"{DIRS['pred']}/{name}_Predictions.xlsx") as w:
        for phase, yt in [("Train", y_train), ("Test", y_test)]:
            p = PRED[name][phase.lower()]
            pd.DataFrame({"Actual": yt, "Predicted": p, "Residual": yt - p,
                          "Abs_Error": np.abs(yt - p)}).to_excel(w, sheet_name=phase, index=False)
print("Predictions saved to", DIRS["pred"])

## 9. Performance analysis
### 9.1 Regression plots (actual vs predicted)

In [ ]:
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
SET_COLORS = {"Training": "#1F4E9E", "Testing": "#B2182B"}
UNIT = "mm"
LAB = LABELS.get(TARGET, TARGET)
# common axis limit for every panel so all models are compared on the same scale
AX_HI = np.ceil(max(y_train.max(), y_test.max(),
                    max(PRED[n][k].max() for n in MODELS for k in ("train", "test"))) * 1.05 / 5) * 5
TICK = 10 if AX_HI > 30 else 5

def reg_panel(fig, spec, yt, yp, color, title, marker="o"):
    """Measured-vs-predicted panel with marginal distributions laid out on its own grid (no overlap)."""
    yt, yp = np.asarray(yt, float), np.asarray(yp, float)
    m = evaluate(yt, yp)
    g = GridSpecFromSubplotSpec(2, 2, subplot_spec=spec, height_ratios=[1, 4.6],
                                width_ratios=[4.6, 1], hspace=0.04, wspace=0.04)
    ax = fig.add_subplot(g[1, 0]); axt = fig.add_subplot(g[0, 0], sharex=ax); axr = fig.add_subplot(g[1, 1], sharey=ax)
    xx = np.linspace(0, AX_HI, 300)

    # error bands, data, 1:1 line
    ax.fill_between(xx, 0.8 * xx, 1.2 * xx, color="#ECECEC", lw=0, zorder=0, label="±20% band")
    ax.fill_between(xx, 0.9 * xx, 1.1 * xx, color="#D4D4D4", lw=0, zorder=0, label="±10% band")
    ax.scatter(yt, yp, s=30, color=color, edgecolor="k", lw=0.4, alpha=0.78, marker=marker, zorder=3,
               label=f"Data (n = {len(yt)})")
    ax.plot([0, AX_HI], [0, AX_HI], "k-", lw=1.3, zorder=4, label="Ideal (1:1)")

    # least-squares fit with 95 % confidence band
    lr = stats.linregress(yt, yp); fit = lr.intercept + lr.slope * xx
    n = len(yt); res = yp - (lr.intercept + lr.slope * yt)
    s_err = np.sqrt(np.sum(res ** 2) / (n - 2)); tq = stats.t.ppf(0.975, n - 2)
    ci = tq * s_err * np.sqrt(1 / n + (xx - yt.mean()) ** 2 / np.sum((yt - yt.mean()) ** 2))
    ax.fill_between(xx, fit - ci, fit + ci, color="#E41A1C", alpha=0.22, lw=0, zorder=4)
    sign = "+" if lr.intercept >= 0 else "−"
    ax.plot(xx, fit, "--", color="#E41A1C", lw=1.7, zorder=5,
            label=f"Fit: y = {lr.slope:.3f}x {sign} {abs(lr.intercept):.3f}")

    # metrics box
    ax.text(0.035, 0.965,
            f"R$^2$ = {m['R2']:.4f}\nRMSE = {m['RMSE']:.4f} {UNIT}\nMAE = {m['MAE']:.4f} {UNIT}\n"
            f"VAF = {m['VAF (%)']:.2f}%\na20-index = {m['a20-index']:.3f}",
            transform=ax.transAxes, va="top", ha="left", fontsize=10.5, zorder=6, linespacing=1.35,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="0.35", lw=0.8, alpha=0.97))

    # axes styling
    ax.set_xlim(0, AX_HI); ax.set_ylim(0, AX_HI)
    for a_ in (ax.xaxis, ax.yaxis):
        a_.set_major_locator(MultipleLocator(TICK)); a_.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(which="both", direction="in", top=True, right=True)
    ax.tick_params(which="major", length=5); ax.tick_params(which="minor", length=2.5)
    ax.grid(True, ls=":", lw=0.6, color="0.75", zorder=0)
    ax.set_xlabel(f"Measured settlement, {LAB} ({UNIT})", labelpad=6)
    ax.set_ylabel(f"Predicted settlement, {LAB} ({UNIT})", labelpad=6)
    ax.legend(loc="lower right", fontsize=8.8, framealpha=0.97, edgecolor="0.6",
              handlelength=1.6, borderpad=0.5, labelspacing=0.35)

    # marginal distributions: top = measured, right = predicted
    bins = np.linspace(0, AX_HI, 26)
    axt.hist(yt, bins=bins, density=True, color=color, alpha=0.35, edgecolor="white", lw=0.5)
    axr.hist(yp, bins=bins, density=True, color=color, alpha=0.35, edgecolor="white", lw=0.5,
             orientation="horizontal")
    kde_t = stats.gaussian_kde(yt); kde_p = stats.gaussian_kde(yp)
    axt.plot(xx, kde_t(xx), color=color, lw=1.6); axr.plot(kde_p(xx), xx, color=color, lw=1.6)
    for a_, side in ((axt, "bottom"), (axr, "left")):
        for sp in a_.spines.values(): sp.set_visible(False)
        a_.spines[side].set_visible(True); a_.spines[side].set_color("0.5")
        a_.tick_params(axis="both", which="both", length=0, left=False, bottom=False, top=False, right=False,
                       labelleft=False, labelbottom=False, labeltop=False, labelright=False)
        a_.grid(False)
    axt.set_ylim(bottom=0); axr.set_xlim(left=0)
    axt.set_title(title, loc="left", fontsize=13, fontweight="bold", pad=6)
    return ax

def reg_figure(rows, fname):
    """rows = list of (model name, row index); 2 panels (training | testing) per row."""
    nr = len(rows)
    fig = plt.figure(figsize=(13, 6.6 * nr))
    gs = GridSpec(nr, 2, figure=fig, left=0.075, right=0.985, top=1 - 0.035 / nr, bottom=0.055 / nr,
                  wspace=0.22, hspace=0.30)
    for r_, (name, k) in enumerate(rows):
        for c_, (phase, key) in enumerate((("Training", "train"), ("Testing", "test"))):
            yt = y_train if key == "train" else y_test
            lab = chr(97 + 2 * k + c_) if nr > 1 else "ab"[c_]
            reg_panel(fig, gs[r_, c_], yt, PRED[name][key], SET_COLORS[phase],
                      f"({lab}) {name} — {phase}", MODEL_MARKERS[name])
    save_fig(fig, "reg", fname)

# ---- Individual model figures (training | testing) ----
for name in MODELS:
    reg_figure([(name, 0)], f"Fig_Regression_{name}")

# ---- Combined grid: all models × (training | testing) — 2 figures per row ----
reg_figure([(name, i) for i, name in enumerate(MODELS)], "Fig_Regression_All_Models")

# ---- Training + testing overlaid, all models (2 × 2) ----
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
hi = max(y.max(), max(PRED[n][k].max() for n in MODELS for k in ["train", "test"])) * 1.05
for ax, name, l in zip(axes.flat, MODELS, "abcd"):
    xx = np.linspace(0, hi, 100)
    ax.fill_between(xx, 0.8 * xx, 1.2 * xx, color="0.92", zorder=0, label="±20% band")
    ax.scatter(y_train, PRED[name]["train"], s=28, c=SET_COLORS["Training"], marker="o", edgecolor="k",
               lw=0.4, alpha=0.65, zorder=3, label=f"Training (R$^2$ = {r2_score(y_train, PRED[name]['train']):.4f})")
    ax.scatter(y_test, PRED[name]["test"], s=40, c=SET_COLORS["Testing"], marker="^", edgecolor="k",
               lw=0.4, alpha=0.85, zorder=4, label=f"Testing (R$^2$ = {r2_score(y_test, PRED[name]['test']):.4f})")
    ax.plot([0, hi], [0, hi], "k-", lw=1.4, zorder=5, label="Ideal (1:1)")
    ax.set_xlim(0, hi); ax.set_ylim(0, hi); ax.set_aspect("equal")
    ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, ls="--", lw=0.5, color="0.85", zorder=0)
    lab = LABELS.get(TARGET, TARGET)
    ax.set_xlabel(f"Actual {lab}"); ax.set_ylabel(f"Predicted {lab}")
    ax.set_title(f"({l}) {name}", loc="left"); ax.legend(loc="upper left", fontsize=9.5, framealpha=0.95)
fig.tight_layout()
save_fig(fig, "reg", "Fig_Regression_Train_Test_Overlay")

### 9.2 Actual vs predicted along samples, Taylor diagram, metric comparison

In [ ]:
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
lab = LABELS.get(TARGET, TARGET)

# ============ (1) Actual vs predicted along samples ============
# (a) one panel per model, training and testing — 2 figures per row
for phase, yt in [("train", y_train), ("test", y_test)]:
    P = "Training" if phase == "train" else "Testing"
    o = np.argsort(yt); xs = np.arange(1, len(yt) + 1); ya = yt[o]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
    for ax, name, l in zip(axes.flat, MODELS, "abcd"):
        yp = PRED[name][phase][o]
        ax.fill_between(xs, ya, yp, color=MODEL_COLORS[name], alpha=0.22, lw=0, label="Residual")
        ax.plot(xs, ya, "-", color="k", lw=2.0, label="Actual", zorder=3)
        ax.plot(xs, yp, MODEL_MARKERS[name], color=MODEL_COLORS[name], ms=4.5, mec="k", mew=0.3,
                alpha=0.85, label="Predicted", zorder=4)
        m = evaluate(yt, PRED[name][phase])
        ax.text(0.03, 0.95, f"R$^2$ = {m['R2']:.4f}\nRMSE = {m['RMSE']:.4f}\nMAE = {m['MAE']:.4f}",
                transform=ax.transAxes, va="top", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.3", alpha=0.95))
        ax.set_title(f"({l}) {name} — {P}", loc="left"); ax.legend(loc="center left", fontsize=9.5)
        ax.set_xlim(0, len(yt) + 1)
        ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())
        ax.grid(True, ls="--", lw=0.5, color="0.85"); ax.set_axisbelow(True)
    for ax in axes[-1]: ax.set_xlabel("Sample number (sorted by actual value)")
    for ax in axes[:, 0]: ax.set_ylabel(lab)
    fig.tight_layout()
    save_fig(fig, "perf", f"Fig_Actual_vs_Predicted_Line_{P}")

# (b) all models together: training | testing — 2 figures in one row
fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
for ax, (phase, yt), l in zip(axes, [("train", y_train), ("test", y_test)], "ab"):
    o = np.argsort(yt); xs = np.arange(1, len(yt) + 1)
    ax.plot(xs, yt[o], "-", color="k", lw=2.4, label="Actual", zorder=5)
    for name in MODELS:
        ax.plot(xs, PRED[name][phase][o], MODEL_MARKERS[name], color=MODEL_COLORS[name], ms=4,
                mec="k", mew=0.25, alpha=0.75, label=name)
    ax.set_xlim(0, len(yt) + 1); ax.set_xlabel("Sample number (sorted by actual value)"); ax.set_ylabel(lab)
    ax.set_title(f"({l}) {'Training' if phase == 'train' else 'Testing'} set", loc="left")
    ax.legend(loc="upper left", fontsize=9.5, markerscale=1.6)
    ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, ls="--", lw=0.5, color="0.85"); ax.set_axisbelow(True)
fig.tight_layout()
save_fig(fig, "perf", "Fig_Actual_vs_Predicted_Line_All_Models")

# ============ (2) Taylor diagram ============
def taylor(fig_, yt, preds, title, pos, smax_factor=1.25):
    ref_std = np.std(yt)
    ax = fig_.add_subplot(pos, projection="polar")
    ax.set_thetamin(0); ax.set_thetamax(90)
    stds = [np.std(p) for p in preds.values()]
    smax = smax_factor * max([ref_std] + stds)
    ax.set_ylim(0, smax)

    # correlation grid (angular)
    rlocs = np.array([0, 0.2, 0.4, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99, 1.0])
    ax.set_thetagrids(np.degrees(np.arccos(rlocs)), [f"{r:g}" for r in rlocs], fontsize=10)
    ax.grid(True, ls=":", lw=0.7, color="0.6")
    ax.tick_params(axis="y", labelsize=10)

    # centred RMSD contours (green, labelled)
    ts, rs = np.meshgrid(np.linspace(0, np.pi / 2, 300), np.linspace(0, smax, 300))
    crmse = np.sqrt(ref_std**2 + rs**2 - 2 * ref_std * rs * np.cos(ts))
    lv = MaxNLocator(5).tick_values(0, crmse.max())[1:]
    cs = ax.contour(ts, rs, crmse, levels=lv, colors="#2CA02C", linestyles="--", linewidths=0.9, alpha=0.8)
    ax.clabel(cs, fmt="%.1f", fontsize=8.5, colors="#2CA02C")

    # reference std arc + reference point
    t = np.linspace(0, np.pi / 2, 200)
    ax.plot(t, np.full_like(t, ref_std), color="k", ls="-.", lw=1.1)
    ax.plot(0, ref_std, "*", color="gold", mec="k", mew=1.0, ms=20, zorder=10, clip_on=False,
            label=f"Actual (SD = {ref_std:.2f})")

    # models
    for name, p in preds.items():
        r = np.corrcoef(yt, p)[0, 1]
        ax.plot(np.arccos(np.clip(r, -1, 1)), np.std(p), MODEL_MARKERS[name], color=MODEL_COLORS[name],
                ms=12, mec="k", mew=1.0, zorder=11, clip_on=False, label=f"{name} (R = {r:.3f})")

    ax.text(np.radians(48), smax * 1.14, "Correlation coefficient (R)", rotation=-42,
            ha="center", va="center", fontsize=12)
    ax.set_xlabel("Standard deviation", labelpad=22, fontsize=12)
    ax.text(np.radians(97), smax * 0.5, "Standard deviation", rotation=90, ha="center", va="center", fontsize=12)
    ax.set_title(title, loc="left", pad=22, fontsize=13)
    return ax

fig = plt.figure(figsize=(15, 7.2))
ax1 = taylor(fig, y_train, {n: PRED[n]["train"] for n in MODELS}, "(a) Training", 121)
ax2 = taylor(fig, y_test, {n: PRED[n]["test"] for n in MODELS}, "(b) Testing", 122)
for ax in (ax1, ax2):
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=10, frameon=False)
fig.text(0.5, 0.005, "Green dashed contours: centred RMS difference", ha="center", fontsize=10, color="#2CA02C")
fig.subplots_adjust(wspace=0.35, bottom=0.2)
save_fig(fig, "perf", "Fig_Taylor_Diagram")

In [ ]:
# Uses bar3d / grouped_bar3d defined in the K-fold CV section (run that cell first)
from matplotlib.ticker import AutoMinorLocator
MODELS_ = list(MODELS)
PHASES_ = ["Training", "Testing"]
PRETTY_M = {"R2": "R$^2$", "RMSE": "RMSE", "MAE": "MAE", "VAF (%)": "VAF (%)", "a20-index": "a20-index",
            "WI": "WI", "LMI": "LMI", "RSR": "RSR", "Adj. R2": "Adj. R$^2$", "MAPE (%)": "MAPE (%)",
            "U95": "U95", "PI": "PI", "R": "R", "|MBE|": "|MBE|"}
LOWER_BETTER = {"RMSE", "MAE", "MAPE (%)"}

def metric_ylim(vals, mtr):
    vals = np.asarray(vals, float); lo_v, hi_v = vals.min(), vals.max()
    if mtr in LOWER_BETTER:
        return 0.0, hi_v * 1.35
    span = hi_v - lo_v if hi_v > lo_v else abs(hi_v) * 0.1
    lo = max(0.0, lo_v - 0.6 * span)
    return lo, hi_v + (hi_v - lo) * 0.35

# ============ (1) Metric comparison — 3-D bars, 2 figures per row ============
show = ["R2", "RMSE", "MAE", "VAF (%)", "a20-index", "WI"]
fig, axes = plt.subplots(3, 2, figsize=(14, 15))
for ax, mtr, l in zip(axes.flat, show, "abcdef"):
    vals = {m: [METRICS[(METRICS.Model == m) & (METRICS.Phase == p)][mtr].values[0] for p in PHASES_]
            for m in MODELS_}
    fmt = "{:.2f}" if mtr == "VAF (%)" else "{:.4f}"
    grouped_bar3d(ax, PHASES_, MODELS_, vals, MODEL_COLORS,
                  metric_ylim(np.concatenate(list(vals.values())), mtr), fmt=fmt, fs=9.5)
    arrow = "↓ lower is better" if mtr in LOWER_BETTER else "↑ higher is better"
    ax.set_ylabel(PRETTY_M[mtr]); ax.set_title(f"({l}) {PRETTY_M[mtr]}   ({arrow})", loc="left")
fig.tight_layout(h_pad=3.5)
save_fig(fig, "perf", "Fig_Metrics_Comparison_Bar")

# ============ (2) Radar chart — training | testing (2 figures in one row) ============
rad = ["R2", "VAF (%)", "WI", "a20-index", "RMSE", "MAE", "MAPE (%)"]   # PI omitted (can be negative)
fig, axes = plt.subplots(1, 2, figsize=(16, 7.4), subplot_kw={"projection": "polar"})
for ax, phase, l in zip(axes, PHASES_, "ab"):
    sub = METRICS[METRICS.Phase == phase].set_index("Model")[rad].copy()
    for c in rad:   # normalise: 1 = best model for that metric
        sub[c] = (sub[c].min() / sub[c]) if c in LOWER_BETTER else (sub[c] / sub[c].max())
    ang = np.linspace(0, 2 * np.pi, len(rad), endpoint=False).tolist(); ang += ang[:1]
    for name, row in sub.iterrows():
        v = row.tolist() + [row.iloc[0]]
        ax.plot(ang, v, "-", marker=MODEL_MARKERS[name], color=MODEL_COLORS[name], lw=2.2, ms=8,
                mec="k", mew=0.6, label=name, zorder=3)
        ax.fill(ang, v, color=MODEL_COLORS[name], alpha=0.10)
    ax.set_theta_offset(np.pi / 2 + np.pi / len(rad))   # no label sits on the horizontal axis
    ax.set_xticks(ang[:-1])
    ax.set_xticklabels([PRETTY_M[r] + (" $^{(inv)}$" if r in LOWER_BETTER else "") for r in rad], fontsize=12)
    ax.tick_params(axis="x", pad=22)
    ax.set_ylim(0, 1.02); ax.set_yticks([0.25, 0.5, 0.75, 1.0])
    ax.set_yticklabels(["0.25", "0.50", "0.75", "1.00"], fontsize=9, color="0.35")
    ax.set_rlabel_position(0)
    ax.grid(True, ls="--", lw=0.6, color="0.7"); ax.spines["polar"].set_color("0.4")
    ax.set_title(f"({l}) {phase}", loc="left", pad=28, fontsize=14)
axes[1].legend(loc="upper center", bbox_to_anchor=(-0.15, -0.08), ncol=4, fontsize=11, frameon=False)
fig.text(0.5, 0.02, "Values normalised so that 1 = best model; (inv) = error metric inverted (min / value)",
         ha="center", fontsize=10, color="0.3")
fig.subplots_adjust(left=0.07, right=0.95, wspace=0.6, bottom=0.17)
save_fig(fig, "perf", "Fig_Radar_Train_Test")

# ============ (3) Rank-based score analysis — training | testing (2 figures in one row) ============
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), gridspec_kw={"wspace": 0.35})
vmax = SCORE.drop(columns=["Model", "Phase", "Score"]).values.max()
for ax, phase, l in zip(axes, PHASES_, "ab"):
    hm = SCORE[SCORE.Phase == phase].set_index("Model").drop(columns=["Phase"])
    total = hm.pop("Score")
    hm = hm.T; hm.index = [PRETTY_M.get(i, i) for i in hm.index]
    hm.loc["Total score"] = total
    mask_total = np.zeros_like(hm.values, dtype=bool); mask_total[-1, :] = True
    sns.heatmap(hm, mask=mask_total, annot=True, fmt=".0f", cmap="YlGnBu", vmin=1, vmax=vmax,
                linewidths=1, linecolor="white", cbar=(l == "b"), annot_kws={"fontsize": 11, "weight": "bold"},
                cbar_kws={"label": "Rank score (higher = better)", "shrink": 0.7}, ax=ax)
    # total-score row highlighted separately
    sns.heatmap(hm, mask=~mask_total, annot=True, fmt=".0f", cmap="Oranges", cbar=False, linewidths=1,
                linecolor="white", annot_kws={"fontsize": 12, "weight": "bold"}, ax=ax)
    ax.axhline(len(hm) - 1, color="k", lw=2)
    best = total.idxmax()
    ax.set_title(f"({l}) {phase} — best: {best}", loc="left", fontsize=13)
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.tick_params(top=False, right=False, bottom=False, left=False)
    plt.setp(ax.get_xticklabels(), rotation=25, ha="right")
save_fig(fig, "perf", "Fig_Score_Ranking_Heatmap")

# ============ (4) Total score ranking — 3-D bars ============
fig, ax = plt.subplots(figsize=(9, 5.5))
vals = {m: [TOTAL.loc[m, "Training"], TOTAL.loc[m, "Testing"], TOTAL.loc[m, "Total score"]] for m in MODELS_}
grouped_bar3d(ax, ["Training", "Testing", "Total"], MODELS_, vals, MODEL_COLORS,
              (0, max(max(v) for v in vals.values()) * 1.3), fmt="{:.0f}", fs=11)
ax.set_ylabel("Rank score (higher = better)")
ax.set_title("Overall ranking — " + ", ".join(f"{m}: #{int(TOTAL.loc[m, 'Rank'])}" for m in TOTAL.index), loc="left",
             fontsize=11.5)
fig.tight_layout()
save_fig(fig, "perf", "Fig_Total_Score_Ranking")

## 10. Error & residual analysis

In [ ]:
from matplotlib.ticker import AutoMinorLocator
lab = LABELS.get(TARGET, TARGET)
PH = [("Training", "train", y_train), ("Testing", "test", y_test)]
SET_COLORS = {"Training": "#1F4E9E", "Testing": "#B2182B"}

def _style(ax, grid_axis="both"):
    ax.grid(True, axis=grid_axis, ls="--", lw=0.5, color="0.85"); ax.set_axisbelow(True)
    ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())

def _box(ax, txt, x=0.97, y=0.95, ha="right"):
    ax.text(x, y, txt, transform=ax.transAxes, ha=ha, va="top", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.3", alpha=0.95), zorder=10)

def _binned_trend(x, y, nb=12):
    """Mean residual in equal-count bins (simple, robust trend line)."""
    o = np.argsort(x); xb = np.array_split(x[o], nb); yb = np.array_split(y[o], nb)
    return np.array([b.mean() for b in xb]), np.array([b.mean() for b in yb])

for P, key, yt in PH:
    # ---------- (a) Residual vs predicted — 2 × 2 ----------
    lim_r = max(np.abs(yt - PRED[n][key]).max() for n in MODELS) * 1.1
    fig, axes = plt.subplots(2, 2, figsize=(13, 10))
    for ax, name, l in zip(axes.flat, MODELS, "abcd"):
        p = PRED[name][key]; res = yt - p; sd = res.std(); mu = res.mean()
        ax.axhspan(-1.96 * sd, 1.96 * sd, color=MODEL_COLORS[name], alpha=0.10, lw=0, label="±1.96 SD (95%)")
        ax.axhline(0, color="k", lw=1.3)
        ax.axhline(1.96 * sd, color="red", ls="--", lw=1.1); ax.axhline(-1.96 * sd, color="red", ls="--", lw=1.1)
        ax.scatter(p, res, s=30, color=MODEL_COLORS[name], edgecolor="k", lw=0.4, alpha=0.8,
                   marker=MODEL_MARKERS[name], zorder=3, label="Residuals")
        bx, by = _binned_trend(p, res)
        ax.plot(bx, by, "-", color="k", lw=1.8, marker="o", ms=4, zorder=4, label="Binned mean trend")
        out = np.mean(np.abs(res) > 1.96 * sd) * 100
        _box(ax, f"Mean = {mu:.3f}\nSD = {sd:.3f}\nOutside 95% band = {out:.1f}%")
        ax.set_ylim(-lim_r, lim_r)
        ax.set_xlabel(f"Predicted {lab}"); ax.set_ylabel("Residual (actual − predicted)")
        ax.set_title(f"({l}) {name} — {P}", loc="left"); ax.legend(loc="lower right", fontsize=9)
        _style(ax)
        # marginal histogram of residuals
        axr = ax.inset_axes([1.01, 0, 0.13, 1], sharey=ax)
        axr.hist(res, bins=25, orientation="horizontal", color=MODEL_COLORS[name], alpha=0.6, edgecolor="white")
        axr.axhline(0, color="k", lw=1); axr.axis("off")
    fig.subplots_adjust(wspace=0.38, hspace=0.3)
    save_fig(fig, "err", f"Fig_Residual_vs_Predicted_{P}")

    # ---------- (c) Absolute error per sample — 2 × 2 ----------
    ymax_ae = max(np.abs(yt - PRED[n][key]).max() for n in MODELS) * 1.12
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
    for ax, name, l in zip(axes.flat, MODELS, "abcd"):
        ae = np.abs(yt - PRED[name][key]); xs = np.arange(1, len(yt) + 1)
        ax.vlines(xs, 0, ae, color=MODEL_COLORS[name], lw=1.2, alpha=0.85)
        ax.scatter(xs, ae, s=14, color=MODEL_COLORS[name], edgecolor="k", lw=0.3, zorder=3)
        ax.axhline(ae.mean(), color="k", ls="--", lw=1.4, label=f"MAE = {ae.mean():.3f}")
        p95 = np.percentile(ae, 95)
        ax.axhline(p95, color="red", ls=":", lw=1.6, label=f"95th percentile = {p95:.3f}")
        ax.set_ylim(0, ymax_ae); ax.set_xlim(0, len(yt) + 1)
        ax.set_title(f"({l}) {name} — {P}", loc="left"); ax.legend(loc="upper right", fontsize=9.5)
        _style(ax, "y")
    for ax in axes[-1]: ax.set_xlabel("Sample number")
    for ax in axes[:, 0]: ax.set_ylabel("Absolute error |actual − predicted|")
    fig.tight_layout()
    save_fig(fig, "err", f"Fig_Absolute_Error_per_Sample_{P}")

# ---------- (b) Residual distribution — training | testing ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=False)
for ax, (P, key, yt), l in zip(axes, PH, "ab"):
    for name in MODELS:
        res = yt - PRED[name][key]
        sns.kdeplot(res, color=MODEL_COLORS[name], lw=2.2, fill=True, alpha=0.12, ax=ax,
                    label=f"{name} (μ = {res.mean():.2f}, σ = {res.std():.2f})")
    ax.axvline(0, color="k", lw=1.2, ls="--")
    ax.set_xlabel("Residual (actual − predicted)"); ax.set_ylabel("Density")
    ax.set_title(f"({l}) {P}", loc="left"); ax.legend(loc="upper right", fontsize=9.5); _style(ax)
fig.tight_layout()
save_fig(fig, "err", "Fig_Residual_Distribution")

# ---------- (d) Residual violin + absolute-error box plot — 2 figures in one row ----------
er = pd.DataFrame([(n, ph, v) for n in MODELS for ph, key, yt in PH for v in (yt - PRED[n][key])],
                  columns=["Model", "Phase", "Residual"])
er["Absolute error"] = er["Residual"].abs()
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.violinplot(data=er, x="Model", y="Residual", hue="Phase", split=True, inner="quartile", cut=0,
               palette=SET_COLORS, linewidth=1.1, saturation=0.85, ax=axes[0])
for c in axes[0].collections: c.set_alpha(0.75)
axes[0].axhline(0, color="k", lw=1.2, ls="--")
axes[0].set_ylabel("Residual (actual − predicted)"); axes[0].set_title("(a) Residual distribution", loc="left")
sns.boxplot(data=er, x="Model", y="Absolute error", hue="Phase", palette=SET_COLORS, showfliers=True, width=0.6,
            linewidth=1.1, flierprops={"marker": "o", "ms": 3, "alpha": 0.5}, ax=axes[1],
            medianprops={"color": "white", "lw": 1.6})
means = er.groupby(["Model", "Phase"], sort=False)["Absolute error"].mean()
for i, m in enumerate(MODELS):
    for j, ph in enumerate(["Training", "Testing"]):
        axes[1].scatter(i + (-0.15 if j == 0 else 0.15), means[(m, ph)], marker="D", s=40, c="gold",
                        edgecolor="k", zorder=5, label="Mean" if (i == 0 and j == 0) else None)
axes[1].set_ylabel("Absolute error"); axes[1].set_title("(b) Absolute error", loc="left")
for ax in axes:
    ax.set_xlabel(""); ax.legend(loc="upper right", fontsize=10); _style(ax, "y")
    ax.xaxis.set_minor_locator(plt.NullLocator()); ax.tick_params(axis="x", top=False)
fig.tight_layout()
save_fig(fig, "err", "Fig_Residual_Violin_Comparison")

# ---------- (e) Q–Q plots of testing residuals with 95% confidence band — 2 × 2 ----------
fig, axes = plt.subplots(2, 2, figsize=(12, 10.5))
for ax, name, l in zip(axes.flat, MODELS, "abcd"):
    res = np.sort(y_test - PRED[name]["test"]); n_ = len(res)
    (osm, osr), (slope, inter, r_) = stats.probplot(res, dist="norm")
    # 95% pointwise confidence envelope (order-statistic standard error)
    pp = (np.arange(1, n_ + 1) - 0.5) / n_
    se = (slope / stats.norm.pdf(osm)) * np.sqrt(pp * (1 - pp) / n_)
    line = inter + slope * osm
    ax.fill_between(osm, line - 1.96 * se, line + 1.96 * se, color=MODEL_COLORS[name], alpha=0.18, lw=0,
                    label="95% confidence band")
    ax.plot(osm, line, "k-", lw=1.5, label="Reference normal line")
    ax.scatter(osm, osr, s=26, color=MODEL_COLORS[name], edgecolor="k", lw=0.4, marker=MODEL_MARKERS[name],
               zorder=3, label="Residuals")
    W, pval = stats.shapiro(res) if n_ <= 5000 else (np.nan, np.nan)
    _box(ax, f"Shapiro–Wilk W = {W:.3f}\np-value = {pval:.3g}\nSkewness = {stats.skew(res):.2f}",
         x=0.03, y=0.95, ha="left")
    ax.set_xlabel("Theoretical quantiles (standard normal)"); ax.set_ylabel("Ordered residuals")
    ax.set_title(f"({l}) {name} — Testing", loc="left"); ax.legend(loc="lower right", fontsize=9.5)
    _style(ax)
fig.tight_layout()
save_fig(fig, "err", "Fig_QQ_Residuals_Testing")

# ---------- (f) Cumulative distribution of absolute error — training | testing ----------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.4))
for ax, (P, key, yt), l in zip(axes, PH, "ab"):
    xmax = 0
    for name in MODELS:
        ae = np.sort(np.abs(yt - PRED[name][key])); cdf = np.arange(1, len(ae) + 1) / len(ae) * 100
        ax.plot(ae, cdf, "-", color=MODEL_COLORS[name], lw=2.2, label=f"{name} (P90 = {np.percentile(ae, 90):.3f})")
        ax.plot(np.percentile(ae, 90), 90, MODEL_MARKERS[name], color=MODEL_COLORS[name], ms=9, mec="k", zorder=5)
        xmax = max(xmax, np.percentile(ae, 99))
    for q in (50, 90, 95):
        ax.axhline(q, color="0.5", ls=":", lw=1); ax.text(xmax * 1.02, q, f"{q}%", va="center", fontsize=9, color="0.4")
    ax.set_xlim(0, xmax * 1.1); ax.set_ylim(0, 101)
    ax.set_xlabel("Absolute error"); ax.set_ylabel("Cumulative percentage of samples (%)")
    ax.set_title(f"({l}) {P}", loc="left"); ax.legend(loc="lower right", fontsize=9.5); _style(ax)
fig.tight_layout()
save_fig(fig, "err", "Fig_Error_CDF")

## 11. Sensitivity analysis of the best model
* **Cosine Amplitude Method (CAM)** — strength of relation between each input and the predicted output
* **Permutation importance** — increase in RMSE when one input is shuffled (testing set, 30 repeats)
* **One-at-a-time (OAT) response curves** — vary one input over its range, others fixed at median
* **SHAP** (optional, if the `shap` package is installed)

In [ ]:
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import Normalize
best_model = MODELS[BEST]
print("Sensitivity analysis for:", BEST)
FL = [LABELS.get(f, f) for f in FEATURES]

# ---------------- computations ----------------
# Cosine Amplitude Method (whole dataset, best-model predictions)
Xn = MinMaxScaler().fit_transform(X)
yhat = best_model.predict(X)
yn = (yhat - yhat.min()) / (yhat.max() - yhat.min())
cam = np.array([np.sum(Xn[:, j] * yn) / np.sqrt(np.sum(Xn[:, j]**2) * np.sum(yn**2)) for j in range(N_FEAT)])

# Permutation importance (testing set)
pi = permutation_importance(best_model, X_test, y_test, scoring="neg_root_mean_squared_error",
                            n_repeats=30, random_state=SEED)
SENS = pd.DataFrame({"Feature": FEATURES, "CAM strength": cam,
                     "Permutation importance (ΔRMSE) mean": pi.importances_mean,
                     "Permutation importance std": pi.importances_std})
SENS["Relative importance (%)"] = 100 * SENS["Permutation importance (ΔRMSE) mean"].clip(lower=0) / \
                                  SENS["Permutation importance (ΔRMSE) mean"].clip(lower=0).sum()
SENS["CAM rank"] = SENS["CAM strength"].rank(ascending=False, method="min").astype(int)
SENS["Permutation rank"] = SENS["Permutation importance (ΔRMSE) mean"].rank(ascending=False, method="min").astype(int)

def _hbar(ax, vals, labels, cmap, xlabel, title, err=None, fmt="{:.3f}", xmax=None):
    o = np.argsort(vals); v = vals[o]
    cols = plt.get_cmap(cmap)(Normalize(v.min() - 0.15 * np.ptp(v), v.max())(v))
    bars = ax.barh(np.array(labels)[o], v, color=cols, edgecolor="k", lw=0.7, height=0.65, zorder=3,
                   xerr=None if err is None else err[o], capsize=4, error_kw={"elinewidth": 1.1, "ecolor": "k"})
    xm = xmax if xmax else (v + (0 if err is None else err[o])).max() * 1.22
    xmin = min(0.0, (v - (0 if err is None else err[o])).min() * 1.6)
    for b, val, e in zip(bars, v, (np.zeros_like(v) if err is None else err[o])):
        xt = max(val, 0) + e + (xm - xmin) * 0.015
        ax.text(xt, b.get_y() + b.get_height() / 2, fmt.format(val), va="center", fontsize=10, fontweight="bold")
    ax.axvline(0, color="k", lw=0.9)
    ax.set_xlim(xmin, xm); ax.set_xlabel(xlabel); ax.set_title(title, loc="left")
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.tick_params(axis="y", which="both", left=False, right=False); ax.tick_params(axis="x", top=False)
    ax.grid(True, axis="x", ls="--", lw=0.5, color="0.85", zorder=0)
    ax.spines[["top", "right"]].set_visible(False)

# ---------------- Diagrams 1 & 2: CAM | permutation importance ----------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))
_hbar(axes[0], cam, FL, "viridis", "Relative strength, R$_{ij}$ (CAM)", f"(a) Cosine amplitude method — {BEST}", xmax=1.12)
_hbar(axes[1], pi.importances_mean, FL, "plasma", "Increase in RMSE when input is permuted",
      f"(b) Permutation importance — {BEST}", err=pi.importances_std)
fig.tight_layout()
save_fig(fig, "sens", f"Fig_Sensitivity_CAM_Permutation_{BEST}")

# ---------------- Diagram 3: relative importance (donut) ----------------
fig, ax = plt.subplots(figsize=(9, 6.5))
pos = SENS[SENS["Relative importance (%)"] > 0].sort_values("Relative importance (%)", ascending=False)
pl = [LABELS.get(f, f) for f in pos.Feature]
cols_ = sns.color_palette("Set2", len(pos))
wedges, _, _ = ax.pie(pos["Relative importance (%)"], labels=None,
                      autopct=lambda p: f"{p:.1f}%" if p >= 3 else "", startangle=90,
                      counterclock=False, colors=cols_, pctdistance=0.80,
                      wedgeprops={"edgecolor": "white", "lw": 2, "width": 0.42},
                      textprops={"fontsize": 10, "fontweight": "bold"})
ax.text(0, 0, f"{BEST}\nrelative\nimportance", ha="center", va="center", fontsize=12)
ax.legend(wedges, [f"{a} ({b:.1f}%)" for a, b in zip(pl, pos["Relative importance (%)"])],
          loc="center left", bbox_to_anchor=(0.98, 0.5), fontsize=11, frameon=False)
ax.set_title("Relative importance of inputs (permutation)", loc="left")
ax.set_aspect("equal")
save_fig(fig, "sens", f"Fig_Relative_Importance_{BEST}")
SENS.sort_values("Permutation rank").round(4)

In [ ]:
# OAT (one-at-a-time) response curves — 2 figures per row
from matplotlib.ticker import AutoMinorLocator, MaxNLocator
ncol_ = 2; nrow_ = int(np.ceil(N_FEAT / ncol_))
NG = 80
fig, axes = plt.subplots(nrow_, ncol_, figsize=(13, 3.9 * nrow_))
oat = {}
q25, med, q75 = np.percentile(X, [25, 50, 75], axis=0)
col = MODEL_COLORS[BEST]
lab_t = LABELS.get(TARGET, TARGET)
for j, (f, ax) in enumerate(zip(FEATURES, axes.flat)):
    grid = np.linspace(X[:, j].min(), X[:, j].max(), NG)
    curves = {}
    for nm, base in [("P25", q25), ("Median", med), ("P75", q75)]:   # other inputs fixed at P25 / median / P75
        Xg = np.tile(base, (NG, 1)); Xg[:, j] = grid
        curves[nm] = best_model.predict(Xg)
    lo = np.minimum(curves["P25"], curves["P75"]); hi = np.maximum(curves["P25"], curves["P75"])
    oat[f] = pd.DataFrame({f: grid, f"Pred_{TARGET}_others_P25": curves["P25"],
                           f"Pred_{TARGET}_others_median": curves["Median"],
                           f"Pred_{TARGET}_others_P75": curves["P75"]})

    ax.fill_between(grid, lo, hi, color=col, alpha=0.18, lw=0, label="Others at P25–P75")
    ax.plot(grid, curves["Median"], "-", color=col, lw=2.4, label="Others at median")
    ax.plot(grid, curves["P25"], ":", color=col, lw=1.2); ax.plot(grid, curves["P75"], ":", color=col, lw=1.2)
    # mark the median value of the varied input
    ym = np.interp(med[j], grid, curves["Median"])
    ax.plot(med[j], ym, "o", ms=8, mfc="white", mec="k", mew=1.3, zorder=5, label="Median of input")
    # rug: where the real data lie
    ylo = min(lo.min(), curves["Median"].min()); yhi = max(hi.max(), curves["Median"].max())
    pad = 0.12 * (yhi - ylo if yhi > ylo else 1)
    ax.set_ylim(ylo - pad, yhi + pad * 1.6)
    ax.plot(X[:, j], np.full(len(X), ylo - pad * 0.55), "|", color="0.35", ms=7, alpha=0.35)
    # range of response
    dS = curves["Median"].max() - curves["Median"].min()
    ax.text(0.97, 0.94, f"ΔS = {dS:.3f}", transform=ax.transAxes, ha="right", va="top", fontsize=10.5,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.3", alpha=0.95))
    ax.set_xlim(grid[0], grid[-1])
    ax.set_xlabel(LABELS.get(f, f)); ax.set_ylabel(f"Predicted {lab_t}")
    ax.set_title(f"({chr(97+j)}) Effect of {LABELS.get(f, f)}", loc="left")
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.xaxis.set_minor_locator(AutoMinorLocator()); ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(True, ls="--", lw=0.5, color="0.85"); ax.set_axisbelow(True)
    if j == 0:
        ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
for ax in axes.flat[N_FEAT:]: ax.axis("off")
fig.tight_layout(h_pad=2)
save_fig(fig, "sens", f"Fig_OAT_Response_Curves_{BEST}")

# OAT summary: response range per input
OAT_SUM = pd.DataFrame({"Feature": FEATURES,
                        "Min input": X.min(0), "Max input": X.max(0),
                        f"Min pred {TARGET}": [oat[f].iloc[:, 2].min() for f in FEATURES],
                        f"Max pred {TARGET}": [oat[f].iloc[:, 2].max() for f in FEATURES]})
OAT_SUM["ΔS (range of response)"] = OAT_SUM[f"Max pred {TARGET}"] - OAT_SUM[f"Min pred {TARGET}"]
OAT_SUM["OAT rank"] = OAT_SUM["ΔS (range of response)"].rank(ascending=False, method="min").astype(int)

with pd.ExcelWriter(f"{DIRS['metric']}/Sensitivity_Analysis_{BEST}.xlsx") as w:
    SENS.to_excel(w, sheet_name="CAM_Permutation", index=False)
    OAT_SUM.to_excel(w, sheet_name="OAT_Summary", index=False)
    pd.DataFrame(pi.importances.T, columns=FEATURES).to_excel(w, sheet_name="Permutation_repeats", index=False)
    for f, d_ in oat.items(): d_.to_excel(w, sheet_name=f"OAT_{f}"[:31], index=False)
OAT_SUM.sort_values("OAT rank").round(4)

In [ ]:
# Optional SHAP analysis (pip install shap)
try:
    import shap
    bg = shap.kmeans(X_train, 25)
    explainer = shap.KernelExplainer(best_model.predict, bg)
    Xs = X_test[:min(150, len(X_test))]
    sv = explainer.shap_values(Xs, nsamples=200)
    names = [LABELS.get(f, f) for f in FEATURES]
    shap.summary_plot(sv, Xs, feature_names=names, show=False); fig = plt.gcf()
    save_fig(fig, "sens", f"Fig_SHAP_Summary_{BEST}")
    shap.summary_plot(sv, Xs, feature_names=names, plot_type="bar", show=False); fig = plt.gcf()
    save_fig(fig, "sens", f"Fig_SHAP_Bar_{BEST}")
    pd.DataFrame(sv, columns=FEATURES).to_excel(f"{DIRS['metric']}/SHAP_values_{BEST}.xlsx", index=False)
except ImportError:
    print("shap not installed — skipped (pip install shap to enable).")

## 12. Summary & how to reload a saved model

In [ ]:
print("Best model:", BEST, HYPER[BEST])
print(METRICS[METRICS.Model == BEST][["Phase", "R2", "RMSE", "MAE", "a20-index", "VAF (%)"]].round(4).to_string(index=False))
print("\nAll outputs saved in:", os.path.abspath(ROOT))
for k, d in DIRS.items():
    print(f"  {d}: {len(os.listdir(d))} files")

# Example: reload and predict (the LSSVM class above must be defined in the session)
loaded = joblib.load(f"{DIRS['models']}/{BEST}.joblib")
print("\nReloaded model check — test R2:", round(r2_score(y_test, loaded.predict(X_test)), 4))